<a href="https://colab.research.google.com/github/minju0236/Hankyung-Bootcamp/blob/main/Day6_9_(260609)_WMS_%EA%B4%80%EB%A6%AC_%EC%8B%9C%EC%8A%A4%ED%85%9C_%ED%99%95%EC%9E%A5_%EC%8B%A4%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%writefile /content/spring-lab/wms-part4/src/main/resources/application.properties

server.port=3100

spring.datasource.url=jdbc:mariadb://localhost:3306/wms_part4
spring.datasource.username=testuser
spring.datasource.password=1234
spring.datasource.driver-class-name=org.mariadb.jdbc.Driver

spring.sql.init.mode=never

spring.thymeleaf.cache=false

Overwriting /content/spring-lab/wms-part4/src/main/resources/application.properties


In [2]:
%%writefile /content/spring-lab/wms-part4/src/main/resources/schema.sql

DROP TABLE IF EXISTS inquiries;
DROP TABLE IF EXISTS notices;
DROP TABLE IF EXISTS outbounds;
DROP TABLE IF EXISTS inventories;
DROP TABLE IF EXISTS inbounds;
DROP TABLE IF EXISTS contracts;
DROP TABLE IF EXISTS users;

CREATE TABLE users (
    id BIGINT NOT NULL AUTO_INCREMENT,
    email VARCHAR(100) NOT NULL,
    password_hash VARCHAR(255) NOT NULL,
    name VARCHAR(50) NOT NULL,
    role VARCHAR(30) NOT NULL,
    status VARCHAR(30) NOT NULL DEFAULT 'ACTIVE',
    created_at DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP,
    updated_at DATETIME NULL,
    PRIMARY KEY (id),
    UNIQUE KEY uk_users_email (email)
);

CREATE TABLE contracts (
    id BIGINT NOT NULL AUTO_INCREMENT,
    customer_id BIGINT NOT NULL,
    product_name VARCHAR(100) NOT NULL,
    quantity INT NOT NULL,
    warehouse_name VARCHAR(100) NOT NULL,
    storage_type VARCHAR(30) NOT NULL,
    request_memo TEXT NULL,
    contract_status VARCHAR(30) NOT NULL DEFAULT 'REQUESTED',
    contract_date DATE NOT NULL,
    created_at DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP,
    updated_at DATETIME NULL,
    PRIMARY KEY (id),
    CONSTRAINT fk_contracts_customer
        FOREIGN KEY (customer_id) REFERENCES users(id)
);

CREATE TABLE inbounds (
    id BIGINT NOT NULL AUTO_INCREMENT,
    contract_id BIGINT NOT NULL,
    received_quantity INT NOT NULL,
    warehouse_name VARCHAR(100) NOT NULL,
    storage_zone VARCHAR(100) NOT NULL,
    pallet_no VARCHAR(50) NULL,
    inbound_status VARCHAR(30) NOT NULL DEFAULT 'REGISTERED',
    inbound_date DATE NULL,
    created_at DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP,
    updated_at DATETIME NULL,
    PRIMARY KEY (id),
    CONSTRAINT fk_inbounds_contract
        FOREIGN KEY (contract_id) REFERENCES contracts(id)
);

CREATE TABLE inventories (
    id BIGINT NOT NULL AUTO_INCREMENT,
    contract_id BIGINT NOT NULL,
    customer_id BIGINT NOT NULL,
    product_name VARCHAR(100) NOT NULL,
    current_quantity INT NOT NULL,
    warehouse_name VARCHAR(100) NOT NULL,
    storage_zone VARCHAR(100) NOT NULL,
    pallet_no VARCHAR(50) NULL,
    inventory_status VARCHAR(30) NOT NULL DEFAULT 'STORED',
    created_at DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP,
    updated_at DATETIME NULL,
    PRIMARY KEY (id),
    CONSTRAINT fk_inventories_contract
        FOREIGN KEY (contract_id) REFERENCES contracts(id),
    CONSTRAINT fk_inventories_customer
        FOREIGN KEY (customer_id) REFERENCES users(id)
);

CREATE TABLE outbounds (
    id BIGINT NOT NULL AUTO_INCREMENT,
    inventory_id BIGINT NOT NULL,
    customer_id BIGINT NOT NULL,
    request_quantity INT NOT NULL,
    desired_date DATE NULL,
    request_memo TEXT NULL,
    outbound_status VARCHAR(30) NOT NULL DEFAULT 'REQUESTED',
    requested_at DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP,
    completed_at DATETIME NULL,
    updated_at DATETIME NULL,
    PRIMARY KEY (id),
    CONSTRAINT fk_outbounds_inventory
        FOREIGN KEY (inventory_id) REFERENCES inventories(id),
    CONSTRAINT fk_outbounds_customer
        FOREIGN KEY (customer_id) REFERENCES users(id)
);

CREATE TABLE notices (
    id BIGINT NOT NULL AUTO_INCREMENT,
    title VARCHAR(200) NOT NULL,
    content TEXT NOT NULL,
    visible BOOLEAN NOT NULL DEFAULT TRUE,
    created_by BIGINT NULL,
    created_at DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP,
    updated_at DATETIME NULL,
    PRIMARY KEY (id),
    CONSTRAINT fk_notices_user
        FOREIGN KEY (created_by) REFERENCES users(id)
);

CREATE TABLE inquiries (
    id BIGINT NOT NULL AUTO_INCREMENT,
    customer_id BIGINT NOT NULL,
    title VARCHAR(200) NOT NULL,
    content TEXT NOT NULL,
    answer_content TEXT NULL,
    inquiry_status VARCHAR(30) NOT NULL DEFAULT 'WAITING',
    created_at DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP,
    answered_at DATETIME NULL,
    PRIMARY KEY (id),
    CONSTRAINT fk_inquiries_customer
        FOREIGN KEY (customer_id) REFERENCES users(id)
);

Writing /content/spring-lab/wms-part4/src/main/resources/schema.sql


In [3]:
%%writefile /content/spring-lab/wms-part4/src/main/resources/data.sql

INSERT INTO users
(email, password_hash, name, role, status)
VALUES
('admin@test.com', '$2a$10$fqNRPb4qcDqZfdVeS2Ms.uYhOzHiYxMbNcl7/.ap2UKTA50vM4Idm', '관리자', 'ROLE_ADMIN', 'ACTIVE'),
('user@test.com', '$2a$10$fqNRPb4qcDqZfdVeS2Ms.uYhOzHiYxMbNcl7/.ap2UKTA50vM4Idm', '고객사용자', 'ROLE_CUSTOMER', 'ACTIVE');

INSERT INTO contracts
(customer_id, product_name, quantity, warehouse_name, storage_type, request_memo, contract_status, contract_date)
VALUES
(2, '유압 실린더', 120, 'A창고', 'NORMAL', '입고 전 외관 검수 필요', 'CONFIRMED', CURRENT_DATE),
(2, '전장 제어 모듈', 80, 'B창고', 'NORMAL', '습기 주의', 'REQUESTED', CURRENT_DATE);

INSERT INTO notices
(title, content, visible, created_by)
VALUES
('WMS 운영 안내', '입고 및 출고 요청은 관리자 확인 후 처리됩니다.', TRUE, 1);

INSERT INTO inquiries
(customer_id, title, content, inquiry_status)
VALUES
(2, '입고 일정 문의', '유압 실린더 입고 예정일을 확인하고 싶습니다.', 'WAITING');

Writing /content/spring-lab/wms-part4/src/main/resources/data.sql


In [4]:
%%writefile /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/domain/AppUser.java

package com.example.wmspart4.domain;

import java.time.LocalDateTime;

public class AppUser {

    private Long id;
    private String email;
    private String passwordHash;
    private String name;
    private String role;
    private String status;
    private LocalDateTime createdAt;
    private LocalDateTime updatedAt;

    public AppUser(Long id, String email, String passwordHash, String name,
                   String role, String status, LocalDateTime createdAt, LocalDateTime updatedAt) {
        this.id = id;
        this.email = email;
        this.passwordHash = passwordHash;
        this.name = name;
        this.role = role;
        this.status = status;
        this.createdAt = createdAt;
        this.updatedAt = updatedAt;
    }

    public Long getId() {
        return id;
    }

    public String getEmail() {
        return email;
    }

    public String getPasswordHash() {
        return passwordHash;
    }

    public String getName() {
        return name;
    }

    public String getRole() {
        return role;
    }

    public String getStatus() {
        return status;
    }

    public LocalDateTime getCreatedAt() {
        return createdAt;
    }

    public LocalDateTime getUpdatedAt() {
        return updatedAt;
    }

    public boolean isActive() {
        return "ACTIVE".equals(status);
    }
}

Writing /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/domain/AppUser.java


In [5]:
%%writefile /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/domain/Contract.java

package com.example.wmspart4.domain;

import java.time.LocalDate;
import java.time.LocalDateTime;

public class Contract {

    private Long id;
    private Long customerId;
    private String customerName;
    private String productName;
    private Integer quantity;
    private String warehouseName;
    private String storageType;
    private String requestMemo;
    private String contractStatus;
    private LocalDate contractDate;
    private LocalDateTime createdAt;
    private LocalDateTime updatedAt;

    public Contract(Long id, Long customerId, String customerName, String productName,
                    Integer quantity, String warehouseName, String storageType,
                    String requestMemo, String contractStatus, LocalDate contractDate,
                    LocalDateTime createdAt, LocalDateTime updatedAt) {
        this.id = id;
        this.customerId = customerId;
        this.customerName = customerName;
        this.productName = productName;
        this.quantity = quantity;
        this.warehouseName = warehouseName;
        this.storageType = storageType;
        this.requestMemo = requestMemo;
        this.contractStatus = contractStatus;
        this.contractDate = contractDate;
        this.createdAt = createdAt;
        this.updatedAt = updatedAt;
    }

    public Long getId() {
        return id;
    }

    public Long getCustomerId() {
        return customerId;
    }

    public String getCustomerName() {
        return customerName;
    }

    public String getProductName() {
        return productName;
    }

    public Integer getQuantity() {
        return quantity;
    }

    public String getWarehouseName() {
        return warehouseName;
    }

    public String getStorageType() {
        return storageType;
    }

    public String getRequestMemo() {
        return requestMemo;
    }

    public String getContractStatus() {
        return contractStatus;
    }

    public LocalDate getContractDate() {
        return contractDate;
    }

    public LocalDateTime getCreatedAt() {
        return createdAt;
    }

    public LocalDateTime getUpdatedAt() {
        return updatedAt;
    }

    public String getStatusLabel() {
        if ("REQUESTED".equals(contractStatus)) {
            return "계약요청";
        }

        if ("CONFIRMED".equals(contractStatus)) {
            return "계약확정";
        }

        if ("CANCELED".equals(contractStatus)) {
            return "계약취소";
        }

        return contractStatus;
    }

    public String getStorageTypeLabel() {
        if ("NORMAL".equals(storageType)) {
            return "일반";
        }

        if ("COLD".equals(storageType)) {
            return "냉장";
        }

        if ("FROZEN".equals(storageType)) {
            return "냉동";
        }

        return storageType;
    }

    public boolean isRequested() {
        return "REQUESTED".equals(contractStatus);
    }

    public boolean isConfirmed() {
        return "CONFIRMED".equals(contractStatus);
    }
}

Writing /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/domain/Contract.java


In [6]:
%%writefile /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/domain/Inbound.java

package com.example.wmspart4.domain;

import java.time.LocalDate;
import java.time.LocalDateTime;

public class Inbound {

    private Long id;
    private Long contractId;
    private String customerName;
    private String productName;
    private Integer receivedQuantity;
    private String warehouseName;
    private String storageZone;
    private String palletNo;
    private String inboundStatus;
    private LocalDate inboundDate;
    private LocalDateTime createdAt;
    private LocalDateTime updatedAt;

    public Inbound(Long id, Long contractId, String customerName, String productName,
                   Integer receivedQuantity, String warehouseName, String storageZone,
                   String palletNo, String inboundStatus, LocalDate inboundDate,
                   LocalDateTime createdAt, LocalDateTime updatedAt) {
        this.id = id;
        this.contractId = contractId;
        this.customerName = customerName;
        this.productName = productName;
        this.receivedQuantity = receivedQuantity;
        this.warehouseName = warehouseName;
        this.storageZone = storageZone;
        this.palletNo = palletNo;
        this.inboundStatus = inboundStatus;
        this.inboundDate = inboundDate;
        this.createdAt = createdAt;
        this.updatedAt = updatedAt;
    }

    public Long getId() {
        return id;
    }

    public Long getContractId() {
        return contractId;
    }

    public String getCustomerName() {
        return customerName;
    }

    public String getProductName() {
        return productName;
    }

    public Integer getReceivedQuantity() {
        return receivedQuantity;
    }

    public String getWarehouseName() {
        return warehouseName;
    }

    public String getStorageZone() {
        return storageZone;
    }

    public String getPalletNo() {
        return palletNo;
    }

    public String getInboundStatus() {
        return inboundStatus;
    }

    public LocalDate getInboundDate() {
        return inboundDate;
    }

    public LocalDateTime getCreatedAt() {
        return createdAt;
    }

    public LocalDateTime getUpdatedAt() {
        return updatedAt;
    }

    public String getStatusLabel() {
        if ("REGISTERED".equals(inboundStatus)) {
            return "입고등록";
        }

        if ("COMPLETED".equals(inboundStatus)) {
            return "입고완료";
        }

        return inboundStatus;
    }

    public boolean isRegistered() {
        return "REGISTERED".equals(inboundStatus);
    }
}

Writing /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/domain/Inbound.java


In [7]:
%%writefile /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/domain/Inventory.java

package com.example.wmspart4.domain;

import java.time.LocalDateTime;

public class Inventory {

    private Long id;
    private Long contractId;
    private Long customerId;
    private String customerName;
    private String productName;
    private Integer currentQuantity;
    private String warehouseName;
    private String storageZone;
    private String palletNo;
    private String inventoryStatus;
    private LocalDateTime updatedAt;

    public Inventory(Long id, Long contractId, Long customerId, String customerName,
                     String productName, Integer currentQuantity, String warehouseName,
                     String storageZone, String palletNo, String inventoryStatus,
                     LocalDateTime updatedAt) {
        this.id = id;
        this.contractId = contractId;
        this.customerId = customerId;
        this.customerName = customerName;
        this.productName = productName;
        this.currentQuantity = currentQuantity;
        this.warehouseName = warehouseName;
        this.storageZone = storageZone;
        this.palletNo = palletNo;
        this.inventoryStatus = inventoryStatus;
        this.updatedAt = updatedAt;
    }

    public Long getId() {
        return id;
    }

    public Long getContractId() {
        return contractId;
    }

    public Long getCustomerId() {
        return customerId;
    }

    public String getCustomerName() {
        return customerName;
    }

    public String getProductName() {
        return productName;
    }

    public Integer getCurrentQuantity() {
        return currentQuantity;
    }

    public String getWarehouseName() {
        return warehouseName;
    }

    public String getStorageZone() {
        return storageZone;
    }

    public String getPalletNo() {
        return palletNo;
    }

    public String getInventoryStatus() {
        return inventoryStatus;
    }

    public LocalDateTime getUpdatedAt() {
        return updatedAt;
    }

    public String getStatusLabel() {
        if ("STORED".equals(inventoryStatus)) {
            return "보관중";
        }

        if ("WAITING_OUTBOUND".equals(inventoryStatus)) {
            return "출고대기";
        }

        if ("EMPTY".equals(inventoryStatus)) {
            return "재고없음";
        }

        return inventoryStatus;
    }
}

Writing /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/domain/Inventory.java


In [8]:
%%writefile /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/domain/Outbound.java

package com.example.wmspart4.domain;

import java.time.LocalDate;
import java.time.LocalDateTime;

public class Outbound {

    private Long id;
    private Long inventoryId;
    private Long customerId;
    private String customerName;
    private String productName;
    private Integer requestQuantity;
    private String outboundStatus;
    private LocalDate desiredDate;
    private LocalDateTime requestedAt;
    private LocalDateTime completedAt;

    public Outbound(Long id, Long inventoryId, Long customerId, String customerName,
                    String productName, Integer requestQuantity, String outboundStatus,
                    LocalDate desiredDate, LocalDateTime requestedAt, LocalDateTime completedAt) {
        this.id = id;
        this.inventoryId = inventoryId;
        this.customerId = customerId;
        this.customerName = customerName;
        this.productName = productName;
        this.requestQuantity = requestQuantity;
        this.outboundStatus = outboundStatus;
        this.desiredDate = desiredDate;
        this.requestedAt = requestedAt;
        this.completedAt = completedAt;
    }

    public Long getId() {
        return id;
    }

    public Long getInventoryId() {
        return inventoryId;
    }

    public Long getCustomerId() {
        return customerId;
    }

    public String getCustomerName() {
        return customerName;
    }

    public String getProductName() {
        return productName;
    }

    public Integer getRequestQuantity() {
        return requestQuantity;
    }

    public String getOutboundStatus() {
        return outboundStatus;
    }

    public LocalDate getDesiredDate() {
        return desiredDate;
    }

    public LocalDateTime getRequestedAt() {
        return requestedAt;
    }

    public LocalDateTime getCompletedAt() {
        return completedAt;
    }

    public String getStatusLabel() {
        if ("REQUESTED".equals(outboundStatus)) {
            return "출고요청";
        }

        if ("COMPLETED".equals(outboundStatus)) {
            return "출고완료";
        }

        return outboundStatus;
    }

    public boolean isRequested() {
        return "REQUESTED".equals(outboundStatus);
    }
}

Writing /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/domain/Outbound.java


In [9]:
%%writefile /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/domain/Notice.java

package com.example.wmspart4.domain;

import java.time.LocalDateTime;

public class Notice {

    private Long id;
    private String title;
    private String content;
    private Boolean visible;
    private LocalDateTime createdAt;

    public Notice(Long id, String title, String content, Boolean visible, LocalDateTime createdAt) {
        this.id = id;
        this.title = title;
        this.content = content;
        this.visible = visible;
        this.createdAt = createdAt;
    }

    public Long getId() {
        return id;
    }

    public String getTitle() {
        return title;
    }

    public String getContent() {
        return content;
    }

    public Boolean getVisible() {
        return visible;
    }

    public LocalDateTime getCreatedAt() {
        return createdAt;
    }
}

Writing /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/domain/Notice.java


In [10]:
%%writefile /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/domain/Inquiry.java

package com.example.wmspart4.domain;

import java.time.LocalDateTime;

public class Inquiry {

    private Long id;
    private Long customerId;
    private String customerName;
    private String title;
    private String content;
    private String answerContent;
    private String inquiryStatus;
    private LocalDateTime createdAt;
    private LocalDateTime answeredAt;

    public Inquiry(Long id, Long customerId, String customerName, String title,
                   String content, String answerContent, String inquiryStatus,
                   LocalDateTime createdAt, LocalDateTime answeredAt) {
        this.id = id;
        this.customerId = customerId;
        this.customerName = customerName;
        this.title = title;
        this.content = content;
        this.answerContent = answerContent;
        this.inquiryStatus = inquiryStatus;
        this.createdAt = createdAt;
        this.answeredAt = answeredAt;
    }

    public Long getId() {
        return id;
    }

    public Long getCustomerId() {
        return customerId;
    }

    public String getCustomerName() {
        return customerName;
    }

    public String getTitle() {
        return title;
    }

    public String getContent() {
        return content;
    }

    public String getAnswerContent() {
        return answerContent;
    }

    public String getInquiryStatus() {
        return inquiryStatus;
    }

    public LocalDateTime getCreatedAt() {
        return createdAt;
    }

    public LocalDateTime getAnsweredAt() {
        return answeredAt;
    }

    public String getStatusLabel() {
        if ("WAITING".equals(inquiryStatus)) {
            return "답변대기";
        }

        if ("ANSWERED".equals(inquiryStatus)) {
            return "답변완료";
        }

        return inquiryStatus;
    }
}

Writing /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/domain/Inquiry.java


In [11]:
%%writefile /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/dto/SignupForm.java

package com.example.wmspart4.dto;

public class SignupForm {

    private String email;
    private String password;
    private String name;
    private String role;

    public String getEmail() {
        return email;
    }

    public String getPassword() {
        return password;
    }

    public String getName() {
        return name;
    }

    public String getRole() {
        return role;
    }

    public void setEmail(String email) {
        this.email = email;
    }

    public void setPassword(String password) {
        this.password = password;
    }

    public void setName(String name) {
        this.name = name;
    }

    public void setRole(String role) {
        this.role = role;
    }
}

Writing /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/dto/SignupForm.java


In [12]:
%%writefile /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/dto/ContractForm.java

package com.example.wmspart4.dto;

public class ContractForm {

    private Long customerId;
    private String productName;
    private Integer quantity;
    private String warehouseName;
    private String storageType;
    private String requestMemo;

    public Long getCustomerId() {
        return customerId;
    }

    public String getProductName() {
        return productName;
    }

    public Integer getQuantity() {
        return quantity;
    }

    public String getWarehouseName() {
        return warehouseName;
    }

    public String getStorageType() {
        return storageType;
    }

    public String getRequestMemo() {
        return requestMemo;
    }

    public void setCustomerId(Long customerId) {
        this.customerId = customerId;
    }

    public void setProductName(String productName) {
        this.productName = productName;
    }

    public void setQuantity(Integer quantity) {
        this.quantity = quantity;
    }

    public void setWarehouseName(String warehouseName) {
        this.warehouseName = warehouseName;
    }

    public void setStorageType(String storageType) {
        this.storageType = storageType;
    }

    public void setRequestMemo(String requestMemo) {
        this.requestMemo = requestMemo;
    }
}

Writing /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/dto/ContractForm.java


In [13]:
%%writefile /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/dto/InboundForm.java

package com.example.wmspart4.dto;

public class InboundForm {

    private Long contractId;
    private Integer receivedQuantity;
    private String warehouseName;
    private String storageZone;
    private String palletNo;

    public Long getContractId() {
        return contractId;
    }

    public Integer getReceivedQuantity() {
        return receivedQuantity;
    }

    public String getWarehouseName() {
        return warehouseName;
    }

    public String getStorageZone() {
        return storageZone;
    }

    public String getPalletNo() {
        return palletNo;
    }

    public void setContractId(Long contractId) {
        this.contractId = contractId;
    }

    public void setReceivedQuantity(Integer receivedQuantity) {
        this.receivedQuantity = receivedQuantity;
    }

    public void setWarehouseName(String warehouseName) {
        this.warehouseName = warehouseName;
    }

    public void setStorageZone(String storageZone) {
        this.storageZone = storageZone;
    }

    public void setPalletNo(String palletNo) {
        this.palletNo = palletNo;
    }
}

Writing /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/dto/InboundForm.java


In [14]:
%%writefile /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/dto/OutboundRequestForm.java

package com.example.wmspart4.dto;

import java.time.LocalDate;

public class OutboundRequestForm {

    private Long inventoryId;
    private Integer requestQuantity;
    private LocalDate desiredDate;
    private String requestMemo;

    public Long getInventoryId() {
        return inventoryId;
    }

    public Integer getRequestQuantity() {
        return requestQuantity;
    }

    public LocalDate getDesiredDate() {
        return desiredDate;
    }

    public String getRequestMemo() {
        return requestMemo;
    }

    public void setInventoryId(Long inventoryId) {
        this.inventoryId = inventoryId;
    }

    public void setRequestQuantity(Integer requestQuantity) {
        this.requestQuantity = requestQuantity;
    }

    public void setDesiredDate(LocalDate desiredDate) {
        this.desiredDate = desiredDate;
    }

    public void setRequestMemo(String requestMemo) {
        this.requestMemo = requestMemo;
    }
}

Writing /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/dto/OutboundRequestForm.java


In [15]:
%%writefile /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/dto/NoticeForm.java

package com.example.wmspart4.dto;

public class NoticeForm {

    private String title;
    private String content;
    private Boolean visible = true;

    public String getTitle() {
        return title;
    }

    public String getContent() {
        return content;
    }

    public Boolean getVisible() {
        return visible;
    }

    public void setTitle(String title) {
        this.title = title;
    }

    public void setContent(String content) {
        this.content = content;
    }

    public void setVisible(Boolean visible) {
        this.visible = visible;
    }
}

Writing /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/dto/NoticeForm.java


In [16]:
%%writefile /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/dto/InquiryForm.java

package com.example.wmspart4.dto;

public class InquiryForm {

    private String title;
    private String content;

    public String getTitle() {
        return title;
    }

    public String getContent() {
        return content;
    }

    public void setTitle(String title) {
        this.title = title;
    }

    public void setContent(String content) {
        this.content = content;
    }
}


Writing /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/dto/InquiryForm.java


In [17]:
%%writefile /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/dto/InquiryAnswerForm.java

package com.example.wmspart4.dto;

public class InquiryAnswerForm {

    private String answerContent;

    public String getAnswerContent() {
        return answerContent;
    }

    public void setAnswerContent(String answerContent) {
        this.answerContent = answerContent;
    }
}

Writing /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/dto/InquiryAnswerForm.java


In [18]:
%%writefile /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/repository/UserRepository.java

package com.example.wmspart4.repository;

import com.example.wmspart4.domain.AppUser;
import org.springframework.jdbc.core.JdbcTemplate;
import org.springframework.stereotype.Repository;

import java.sql.Timestamp;
import java.util.List;
import java.util.Optional;

@Repository
public class UserRepository {

    private final JdbcTemplate jdbcTemplate;

    public UserRepository(JdbcTemplate jdbcTemplate) {
        this.jdbcTemplate = jdbcTemplate;
    }

    public void save(String email, String passwordHash, String name, String role) {
        String sql = """
                INSERT INTO users
                (email, password_hash, name, role, status)
                VALUES (?, ?, ?, ?, 'ACTIVE')
                """;

        jdbcTemplate.update(sql, email, passwordHash, name, role);
    }

    public Optional<AppUser> findByEmail(String email) {
        String sql = """
                SELECT id, email, password_hash, name, role, status, created_at, updated_at
                FROM users
                WHERE email = ?
                """;

        List<AppUser> users = jdbcTemplate.query(sql, (rs, rowNum) -> {
            Timestamp updatedAt = rs.getTimestamp("updated_at");

            return new AppUser(
                    rs.getLong("id"),
                    rs.getString("email"),
                    rs.getString("password_hash"),
                    rs.getString("name"),
                    rs.getString("role"),
                    rs.getString("status"),
                    rs.getTimestamp("created_at").toLocalDateTime(),
                    updatedAt == null ? null : updatedAt.toLocalDateTime()
            );
        }, email);

        return users.stream().findFirst();
    }

    public Optional<AppUser> findById(Long id) {
        String sql = """
                SELECT id, email, password_hash, name, role, status, created_at, updated_at
                FROM users
                WHERE id = ?
                """;

        List<AppUser> users = jdbcTemplate.query(sql, (rs, rowNum) -> {
            Timestamp updatedAt = rs.getTimestamp("updated_at");

            return new AppUser(
                    rs.getLong("id"),
                    rs.getString("email"),
                    rs.getString("password_hash"),
                    rs.getString("name"),
                    rs.getString("role"),
                    rs.getString("status"),
                    rs.getTimestamp("created_at").toLocalDateTime(),
                    updatedAt == null ? null : updatedAt.toLocalDateTime()
            );
        }, id);

        return users.stream().findFirst();
    }

    public List<AppUser> findCustomers() {
        String sql = """
                SELECT id, email, password_hash, name, role, status, created_at, updated_at
                FROM users
                WHERE role = 'ROLE_CUSTOMER'
                ORDER BY id DESC
                """;

        return jdbcTemplate.query(sql, (rs, rowNum) -> {
            Timestamp updatedAt = rs.getTimestamp("updated_at");

            return new AppUser(
                    rs.getLong("id"),
                    rs.getString("email"),
                    rs.getString("password_hash"),
                    rs.getString("name"),
                    rs.getString("role"),
                    rs.getString("status"),
                    rs.getTimestamp("created_at").toLocalDateTime(),
                    updatedAt == null ? null : updatedAt.toLocalDateTime()
            );
        });
    }

    public boolean existsByEmail(String email) {
        Integer count = jdbcTemplate.queryForObject(
                "SELECT COUNT(*) FROM users WHERE email = ?",
                Integer.class,
                email
        );

        return count != null && count > 0;
    }

    public long countAll() {
        Long count = jdbcTemplate.queryForObject("SELECT COUNT(*) FROM users", Long.class);
        return count == null ? 0 : count;
    }
}


Writing /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/repository/UserRepository.java


In [19]:
%%writefile /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/repository/ContractRepository.java

package com.example.wmspart4.repository;

import com.example.wmspart4.domain.Contract;
import com.example.wmspart4.dto.ContractForm;
import org.springframework.jdbc.core.JdbcTemplate;
import org.springframework.stereotype.Repository;

import java.sql.Timestamp;
import java.util.List;
import java.util.Optional;

@Repository
public class ContractRepository {

    private final JdbcTemplate jdbcTemplate;

    public ContractRepository(JdbcTemplate jdbcTemplate) {
        this.jdbcTemplate = jdbcTemplate;
    }

    public void save(ContractForm form) {
        String sql = """
                INSERT INTO contracts
                (customer_id, product_name, quantity, warehouse_name,
                 storage_type, request_memo, contract_status, contract_date)
                VALUES (?, ?, ?, ?, ?, ?, 'REQUESTED', CURRENT_DATE)
                """;

        jdbcTemplate.update(
                sql,
                form.getCustomerId(),
                form.getProductName(),
                form.getQuantity(),
                form.getWarehouseName(),
                form.getStorageType(),
                form.getRequestMemo()
        );
    }

    public List<Contract> findAll() {
        String sql = """
                SELECT c.id, c.customer_id, u.name AS customer_name,
                       c.product_name, c.quantity, c.warehouse_name, c.storage_type,
                       c.request_memo, c.contract_status, c.contract_date,
                       c.created_at, c.updated_at
                FROM contracts c
                JOIN users u ON c.customer_id = u.id
                ORDER BY c.id DESC
                """;

        return jdbcTemplate.query(sql, (rs, rowNum) -> mapContract(rs));
    }

    public List<Contract> findByCustomerId(Long customerId) {
        String sql = """
                SELECT c.id, c.customer_id, u.name AS customer_name,
                       c.product_name, c.quantity, c.warehouse_name, c.storage_type,
                       c.request_memo, c.contract_status, c.contract_date,
                       c.created_at, c.updated_at
                FROM contracts c
                JOIN users u ON c.customer_id = u.id
                WHERE c.customer_id = ?
                ORDER BY c.id DESC
                """;

        return jdbcTemplate.query(sql, (rs, rowNum) -> mapContract(rs), customerId);
    }

    public Optional<Contract> findById(Long id) {
        String sql = """
                SELECT c.id, c.customer_id, u.name AS customer_name,
                       c.product_name, c.quantity, c.warehouse_name, c.storage_type,
                       c.request_memo, c.contract_status, c.contract_date,
                       c.created_at, c.updated_at
                FROM contracts c
                JOIN users u ON c.customer_id = u.id
                WHERE c.id = ?
                """;

        List<Contract> contracts = jdbcTemplate.query(sql, (rs, rowNum) -> mapContract(rs), id);
        return contracts.stream().findFirst();
    }

    public void updateStatus(Long id, String status) {
        String sql = """
                UPDATE contracts
                SET contract_status = ?,
                    updated_at = CURRENT_TIMESTAMP
                WHERE id = ?
                """;

        jdbcTemplate.update(sql, status, id);
    }

    public long countAll() {
        Long count = jdbcTemplate.queryForObject("SELECT COUNT(*) FROM contracts", Long.class);
        return count == null ? 0 : count;
    }

    private Contract mapContract(java.sql.ResultSet rs) throws java.sql.SQLException {
        Timestamp updatedAt = rs.getTimestamp("updated_at");

        return new Contract(
                rs.getLong("id"),
                rs.getLong("customer_id"),
                rs.getString("customer_name"),
                rs.getString("product_name"),
                rs.getInt("quantity"),
                rs.getString("warehouse_name"),
                rs.getString("storage_type"),
                rs.getString("request_memo"),
                rs.getString("contract_status"),
                rs.getDate("contract_date").toLocalDate(),
                rs.getTimestamp("created_at").toLocalDateTime(),
                updatedAt == null ? null : updatedAt.toLocalDateTime()
        );
    }
}


Writing /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/repository/ContractRepository.java


In [20]:
%%writefile /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/repository/InboundRepository.java

package com.example.wmspart4.repository;

import com.example.wmspart4.domain.Inbound;
import com.example.wmspart4.dto.InboundForm;
import org.springframework.jdbc.core.JdbcTemplate;
import org.springframework.stereotype.Repository;

import java.sql.Date;
import java.sql.Timestamp;
import java.util.List;
import java.util.Optional;

@Repository
public class InboundRepository {

    private final JdbcTemplate jdbcTemplate;

    public InboundRepository(JdbcTemplate jdbcTemplate) {
        this.jdbcTemplate = jdbcTemplate;
    }

    public void save(InboundForm form) {
        String sql = """
                INSERT INTO inbounds
                (contract_id, received_quantity, warehouse_name, storage_zone,
                 pallet_no, inbound_status)
                VALUES (?, ?, ?, ?, ?, 'REGISTERED')
                """;

        jdbcTemplate.update(
                sql,
                form.getContractId(),
                form.getReceivedQuantity(),
                form.getWarehouseName(),
                form.getStorageZone(),
                form.getPalletNo()
        );
    }

    public List<Inbound> findAll() {
        String sql = """
                SELECT i.id, i.contract_id, u.name AS customer_name, c.product_name,
                       i.received_quantity, i.warehouse_name, i.storage_zone,
                       i.pallet_no, i.inbound_status, i.inbound_date,
                       i.created_at, i.updated_at
                FROM inbounds i
                JOIN contracts c ON i.contract_id = c.id
                JOIN users u ON c.customer_id = u.id
                ORDER BY i.id DESC
                """;

        return jdbcTemplate.query(sql, (rs, rowNum) -> mapInbound(rs));
    }

    public Optional<Inbound> findById(Long id) {
        String sql = """
                SELECT i.id, i.contract_id, u.name AS customer_name, c.product_name,
                       i.received_quantity, i.warehouse_name, i.storage_zone,
                       i.pallet_no, i.inbound_status, i.inbound_date,
                       i.created_at, i.updated_at
                FROM inbounds i
                JOIN contracts c ON i.contract_id = c.id
                JOIN users u ON c.customer_id = u.id
                WHERE i.id = ?
                """;

        List<Inbound> inbounds = jdbcTemplate.query(sql, (rs, rowNum) -> mapInbound(rs), id);
        return inbounds.stream().findFirst();
    }

    public void complete(Long id) {
        String sql = """
                UPDATE inbounds
                SET inbound_status = 'COMPLETED',
                    inbound_date = CURRENT_DATE,
                    updated_at = CURRENT_TIMESTAMP
                WHERE id = ?
                """;

        jdbcTemplate.update(sql, id);
    }

    public long countAll() {
        Long count = jdbcTemplate.queryForObject("SELECT COUNT(*) FROM inbounds", Long.class);
        return count == null ? 0 : count;
    }

    private Inbound mapInbound(java.sql.ResultSet rs) throws java.sql.SQLException {
        Date inboundDate = rs.getDate("inbound_date");
        Timestamp updatedAt = rs.getTimestamp("updated_at");

        return new Inbound(
                rs.getLong("id"),
                rs.getLong("contract_id"),
                rs.getString("customer_name"),
                rs.getString("product_name"),
                rs.getInt("received_quantity"),
                rs.getString("warehouse_name"),
                rs.getString("storage_zone"),
                rs.getString("pallet_no"),
                rs.getString("inbound_status"),
                inboundDate == null ? null : inboundDate.toLocalDate(),
                rs.getTimestamp("created_at").toLocalDateTime(),
                updatedAt == null ? null : updatedAt.toLocalDateTime()
        );
    }
}

Writing /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/repository/InboundRepository.java


In [21]:
%%writefile /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/repository/InventoryRepository.java

package com.example.wmspart4.repository;

import com.example.wmspart4.domain.Contract;
import com.example.wmspart4.domain.Inventory;
import com.example.wmspart4.domain.Inbound;
import org.springframework.jdbc.core.JdbcTemplate;
import org.springframework.stereotype.Repository;

import java.sql.Timestamp;
import java.util.List;
import java.util.Optional;

@Repository
public class InventoryRepository {

    private final JdbcTemplate jdbcTemplate;

    public InventoryRepository(JdbcTemplate jdbcTemplate) {
        this.jdbcTemplate = jdbcTemplate;
    }

    public void createFromInbound(Inbound inbound, Contract contract) {
        String sql = """
                INSERT INTO inventories
                (contract_id, customer_id, product_name, current_quantity,
                 warehouse_name, storage_zone, pallet_no, inventory_status)
                VALUES (?, ?, ?, ?, ?, ?, ?, 'STORED')
                """;

        jdbcTemplate.update(
                sql,
                contract.getId(),
                contract.getCustomerId(),
                contract.getProductName(),
                inbound.getReceivedQuantity(),
                inbound.getWarehouseName(),
                inbound.getStorageZone(),
                inbound.getPalletNo()
        );
    }

    public List<Inventory> findAll() {
        String sql = """
                SELECT i.id, i.contract_id, i.customer_id, u.name AS customer_name,
                       i.product_name, i.current_quantity, i.warehouse_name,
                       i.storage_zone, i.pallet_no, i.inventory_status, i.updated_at
                FROM inventories i
                JOIN users u ON i.customer_id = u.id
                ORDER BY i.id DESC
                """;

        return jdbcTemplate.query(sql, (rs, rowNum) -> mapInventory(rs));
    }

    public List<Inventory> findByCustomerId(Long customerId) {
        String sql = """
                SELECT i.id, i.contract_id, i.customer_id, u.name AS customer_name,
                       i.product_name, i.current_quantity, i.warehouse_name,
                       i.storage_zone, i.pallet_no, i.inventory_status, i.updated_at
                FROM inventories i
                JOIN users u ON i.customer_id = u.id
                WHERE i.customer_id = ?
                ORDER BY i.id DESC
                """;

        return jdbcTemplate.query(sql, (rs, rowNum) -> mapInventory(rs), customerId);
    }

    public Optional<Inventory> findById(Long id) {
        String sql = """
                SELECT i.id, i.contract_id, i.customer_id, u.name AS customer_name,
                       i.product_name, i.current_quantity, i.warehouse_name,
                       i.storage_zone, i.pallet_no, i.inventory_status, i.updated_at
                FROM inventories i
                JOIN users u ON i.customer_id = u.id
                WHERE i.id = ?
                """;

        List<Inventory> inventories = jdbcTemplate.query(sql, (rs, rowNum) -> mapInventory(rs), id);
        return inventories.stream().findFirst();
    }

    public void decreaseQuantity(Long id, int quantity) {
        String sql = """
                UPDATE inventories
                SET current_quantity = current_quantity - ?,
                    inventory_status = CASE
                        WHEN current_quantity - ? <= 0 THEN 'EMPTY'
                        ELSE 'STORED'
                    END,
                    updated_at = CURRENT_TIMESTAMP
                WHERE id = ?
                """;

        jdbcTemplate.update(sql, quantity, quantity, id);
    }

    public long countAll() {
        Long count = jdbcTemplate.queryForObject("SELECT COUNT(*) FROM inventories", Long.class);
        return count == null ? 0 : count;
    }

    private Inventory mapInventory(java.sql.ResultSet rs) throws java.sql.SQLException {
        Timestamp updatedAt = rs.getTimestamp("updated_at");

        return new Inventory(
                rs.getLong("id"),
                rs.getLong("contract_id"),
                rs.getLong("customer_id"),
                rs.getString("customer_name"),
                rs.getString("product_name"),
                rs.getInt("current_quantity"),
                rs.getString("warehouse_name"),
                rs.getString("storage_zone"),
                rs.getString("pallet_no"),
                rs.getString("inventory_status"),
                updatedAt == null ? null : updatedAt.toLocalDateTime()
        );
    }
}


Writing /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/repository/InventoryRepository.java


In [22]:
%%writefile /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/repository/OutboundRepository.java

package com.example.wmspart4.repository;

import com.example.wmspart4.domain.Outbound;
import com.example.wmspart4.dto.OutboundRequestForm;
import org.springframework.jdbc.core.JdbcTemplate;
import org.springframework.stereotype.Repository;

import java.sql.Date;
import java.sql.Timestamp;
import java.util.List;
import java.util.Optional;

@Repository
public class OutboundRepository {

    private final JdbcTemplate jdbcTemplate;

    public OutboundRepository(JdbcTemplate jdbcTemplate) {
        this.jdbcTemplate = jdbcTemplate;
    }

    public void save(Long customerId, OutboundRequestForm form) {
        String sql = """
                INSERT INTO outbounds
                (inventory_id, customer_id, request_quantity, desired_date,
                 request_memo, outbound_status)
                VALUES (?, ?, ?, ?, ?, 'REQUESTED')
                """;

        jdbcTemplate.update(
                sql,
                form.getInventoryId(),
                customerId,
                form.getRequestQuantity(),
                form.getDesiredDate(),
                form.getRequestMemo()
        );
    }

    public List<Outbound> findAll() {
        String sql = """
                SELECT o.id, o.inventory_id, o.customer_id, u.name AS customer_name,
                       i.product_name, o.request_quantity, o.outbound_status,
                       o.desired_date, o.requested_at, o.completed_at
                FROM outbounds o
                JOIN inventories i ON o.inventory_id = i.id
                JOIN users u ON o.customer_id = u.id
                ORDER BY o.id DESC
                """;

        return jdbcTemplate.query(sql, (rs, rowNum) -> mapOutbound(rs));
    }

    public List<Outbound> findByCustomerId(Long customerId) {
        String sql = """
                SELECT o.id, o.inventory_id, o.customer_id, u.name AS customer_name,
                       i.product_name, o.request_quantity, o.outbound_status,
                       o.desired_date, o.requested_at, o.completed_at
                FROM outbounds o
                JOIN inventories i ON o.inventory_id = i.id
                JOIN users u ON o.customer_id = u.id
                WHERE o.customer_id = ?
                ORDER BY o.id DESC
                """;

        return jdbcTemplate.query(sql, (rs, rowNum) -> mapOutbound(rs), customerId);
    }

    public Optional<Outbound> findById(Long id) {
        String sql = """
                SELECT o.id, o.inventory_id, o.customer_id, u.name AS customer_name,
                       i.product_name, o.request_quantity, o.outbound_status,
                       o.desired_date, o.requested_at, o.completed_at
                FROM outbounds o
                JOIN inventories i ON o.inventory_id = i.id
                JOIN users u ON o.customer_id = u.id
                WHERE o.id = ?
                """;

        List<Outbound> outbounds = jdbcTemplate.query(sql, (rs, rowNum) -> mapOutbound(rs), id);
        return outbounds.stream().findFirst();
    }

    public void complete(Long id) {
        String sql = """
                UPDATE outbounds
                SET outbound_status = 'COMPLETED',
                    completed_at = CURRENT_TIMESTAMP,
                    updated_at = CURRENT_TIMESTAMP
                WHERE id = ?
                """;

        jdbcTemplate.update(sql, id);
    }

    public long countAll() {
        Long count = jdbcTemplate.queryForObject("SELECT COUNT(*) FROM outbounds", Long.class);
        return count == null ? 0 : count;
    }

    private Outbound mapOutbound(java.sql.ResultSet rs) throws java.sql.SQLException {
        Date desiredDate = rs.getDate("desired_date");
        Timestamp completedAt = rs.getTimestamp("completed_at");

        return new Outbound(
                rs.getLong("id"),
                rs.getLong("inventory_id"),
                rs.getLong("customer_id"),
                rs.getString("customer_name"),
                rs.getString("product_name"),
                rs.getInt("request_quantity"),
                rs.getString("outbound_status"),
                desiredDate == null ? null : desiredDate.toLocalDate(),
                rs.getTimestamp("requested_at").toLocalDateTime(),
                completedAt == null ? null : completedAt.toLocalDateTime()
        );
    }
}


Writing /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/repository/OutboundRepository.java


In [23]:
%%writefile /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/repository/NoticeRepository.java

package com.example.wmspart4.repository;

import com.example.wmspart4.domain.Notice;
import com.example.wmspart4.dto.NoticeForm;
import org.springframework.jdbc.core.JdbcTemplate;
import org.springframework.stereotype.Repository;

import java.util.List;

@Repository
public class NoticeRepository {

    private final JdbcTemplate jdbcTemplate;

    public NoticeRepository(JdbcTemplate jdbcTemplate) {
        this.jdbcTemplate = jdbcTemplate;
    }

    public void save(Long createdBy, NoticeForm form) {
        String sql = """
                INSERT INTO notices
                (title, content, visible, created_by)
                VALUES (?, ?, ?, ?)
                """;

        jdbcTemplate.update(sql, form.getTitle(), form.getContent(), form.getVisible(), createdBy);
    }

    public List<Notice> findVisible() {
        String sql = """
                SELECT id, title, content, visible, created_at
                FROM notices
                WHERE visible = TRUE
                ORDER BY id DESC
                """;

        return jdbcTemplate.query(sql, (rs, rowNum) -> new Notice(
                rs.getLong("id"),
                rs.getString("title"),
                rs.getString("content"),
                rs.getBoolean("visible"),
                rs.getTimestamp("created_at").toLocalDateTime()
        ));
    }

    public List<Notice> findAll() {
        String sql = """
                SELECT id, title, content, visible, created_at
                FROM notices
                ORDER BY id DESC
                """;

        return jdbcTemplate.query(sql, (rs, rowNum) -> new Notice(
                rs.getLong("id"),
                rs.getString("title"),
                rs.getString("content"),
                rs.getBoolean("visible"),
                rs.getTimestamp("created_at").toLocalDateTime()
        ));
    }
}

Writing /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/repository/NoticeRepository.java


In [24]:
%%writefile /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/repository/InquiryRepository.java

package com.example.wmspart4.repository;

import com.example.wmspart4.domain.Inquiry;
import com.example.wmspart4.dto.InquiryForm;
import org.springframework.jdbc.core.JdbcTemplate;
import org.springframework.stereotype.Repository;

import java.sql.Timestamp;
import java.util.List;

@Repository
public class InquiryRepository {

    private final JdbcTemplate jdbcTemplate;

    public InquiryRepository(JdbcTemplate jdbcTemplate) {
        this.jdbcTemplate = jdbcTemplate;
    }

    public void save(Long customerId, InquiryForm form) {
        String sql = """
                INSERT INTO inquiries
                (customer_id, title, content, inquiry_status)
                VALUES (?, ?, ?, 'WAITING')
                """;

        jdbcTemplate.update(sql, customerId, form.getTitle(), form.getContent());
    }

    public List<Inquiry> findAll() {
        String sql = """
                SELECT i.id, i.customer_id, u.name AS customer_name, i.title, i.content,
                       i.answer_content, i.inquiry_status, i.created_at, i.answered_at
                FROM inquiries i
                JOIN users u ON i.customer_id = u.id
                ORDER BY i.id DESC
                """;

        return jdbcTemplate.query(sql, (rs, rowNum) -> mapInquiry(rs));
    }

    public List<Inquiry> findByCustomerId(Long customerId) {
        String sql = """
                SELECT i.id, i.customer_id, u.name AS customer_name, i.title, i.content,
                       i.answer_content, i.inquiry_status, i.created_at, i.answered_at
                FROM inquiries i
                JOIN users u ON i.customer_id = u.id
                WHERE i.customer_id = ?
                ORDER BY i.id DESC
                """;

        return jdbcTemplate.query(sql, (rs, rowNum) -> mapInquiry(rs), customerId);
    }

    public void answer(Long id, String answerContent) {
        String sql = """
                UPDATE inquiries
                SET answer_content = ?,
                    inquiry_status = 'ANSWERED',
                    answered_at = CURRENT_TIMESTAMP
                WHERE id = ?
                """;

        jdbcTemplate.update(sql, answerContent, id);
    }

    private Inquiry mapInquiry(java.sql.ResultSet rs) throws java.sql.SQLException {
        Timestamp answeredAt = rs.getTimestamp("answered_at");

        return new Inquiry(
                rs.getLong("id"),
                rs.getLong("customer_id"),
                rs.getString("customer_name"),
                rs.getString("title"),
                rs.getString("content"),
                rs.getString("answer_content"),
                rs.getString("inquiry_status"),
                rs.getTimestamp("created_at").toLocalDateTime(),
                answeredAt == null ? null : answeredAt.toLocalDateTime()
        );
    }
}


Writing /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/repository/InquiryRepository.java


In [25]:
%%writefile /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/service/CustomUserDetailsService.java

package com.example.wmspart4.service;

import com.example.wmspart4.domain.AppUser;
import com.example.wmspart4.repository.UserRepository;
import org.springframework.security.core.authority.SimpleGrantedAuthority;
import org.springframework.security.core.userdetails.User;
import org.springframework.security.core.userdetails.UserDetails;
import org.springframework.security.core.userdetails.UserDetailsService;
import org.springframework.security.core.userdetails.UsernameNotFoundException;
import org.springframework.stereotype.Service;

import java.util.List;

@Service
public class CustomUserDetailsService implements UserDetailsService {

    private final UserRepository userRepository;

    public CustomUserDetailsService(UserRepository userRepository) {
        this.userRepository = userRepository;
    }

    @Override
    public UserDetails loadUserByUsername(String email) throws UsernameNotFoundException {
        AppUser appUser = userRepository.findByEmail(email)
                .orElseThrow(() -> new UsernameNotFoundException("사용자를 찾을 수 없습니다."));

        if (!appUser.isActive()) {
            throw new UsernameNotFoundException("비활성 사용자입니다.");
        }

        return new User(
                appUser.getEmail(),
                appUser.getPasswordHash(),
                List.of(new SimpleGrantedAuthority(appUser.getRole()))
        );
    }
}

Writing /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/service/CustomUserDetailsService.java


In [26]:
%%writefile /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/service/UserService.java

package com.example.wmspart4.service;

import com.example.wmspart4.domain.AppUser;
import com.example.wmspart4.dto.SignupForm;
import com.example.wmspart4.repository.UserRepository;
import org.springframework.security.crypto.password.PasswordEncoder;
import org.springframework.stereotype.Service;

import java.util.List;

@Service
public class UserService {

    private final UserRepository userRepository;
    private final PasswordEncoder passwordEncoder;

    public UserService(UserRepository userRepository, PasswordEncoder passwordEncoder) {
        this.userRepository = userRepository;
        this.passwordEncoder = passwordEncoder;
    }

    public void signup(SignupForm form) {
        if (isBlank(form.getEmail()) || isBlank(form.getPassword()) || isBlank(form.getName())) {
            throw new IllegalArgumentException("이메일, 비밀번호, 이름을 모두 입력해야 합니다.");
        }

        if (!"ROLE_CUSTOMER".equals(form.getRole()) && !"ROLE_ADMIN".equals(form.getRole())) {
            throw new IllegalArgumentException("사용자 유형을 선택해야 합니다.");
        }

        String email = form.getEmail().trim();

        if (userRepository.existsByEmail(email)) {
            throw new IllegalArgumentException("이미 사용 중인 이메일입니다.");
        }

        userRepository.save(
                email,
                passwordEncoder.encode(form.getPassword()),
                form.getName().trim(),
                form.getRole()
        );
    }

    public AppUser findByEmail(String email) {
        return userRepository.findByEmail(email)
                .orElseThrow(() -> new IllegalArgumentException("사용자를 찾을 수 없습니다."));
    }

    public List<AppUser> findCustomers() {
        return userRepository.findCustomers();
    }

    public long countAllUsers() {
        return userRepository.countAll();
    }

    private boolean isBlank(String value) {
        return value == null || value.trim().isEmpty();
    }
}

Writing /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/service/UserService.java


In [27]:
%%writefile /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/service/ContractService.java

package com.example.wmspart4.service;

import com.example.wmspart4.domain.Contract;
import com.example.wmspart4.dto.ContractForm;
import com.example.wmspart4.repository.ContractRepository;
import org.springframework.stereotype.Service;

import java.util.List;

@Service
public class ContractService {

    private final ContractRepository contractRepository;

    public ContractService(ContractRepository contractRepository) {
        this.contractRepository = contractRepository;
    }

    public List<Contract> findAll() {
        return contractRepository.findAll();
    }

    public List<Contract> findByCustomerId(Long customerId) {
        return contractRepository.findByCustomerId(customerId);
    }

    public void create(ContractForm form) {
        if (form.getCustomerId() == null) {
            throw new IllegalArgumentException("고객을 선택해야 합니다.");
        }

        if (isBlank(form.getProductName())) {
            throw new IllegalArgumentException("상품명을 입력해야 합니다.");
        }

        if (form.getQuantity() == null || form.getQuantity() <= 0) {
            throw new IllegalArgumentException("수량은 1 이상이어야 합니다.");
        }

        if (isBlank(form.getWarehouseName())) {
            throw new IllegalArgumentException("창고명을 입력해야 합니다.");
        }

        if (isBlank(form.getStorageType())) {
            throw new IllegalArgumentException("보관유형을 선택해야 합니다.");
        }

        contractRepository.save(form);
    }

    public void confirm(Long id) {
        Contract contract = findById(id);

        if (!contract.isRequested()) {
            throw new IllegalStateException("계약요청 상태에서만 확정할 수 있습니다.");
        }

        contractRepository.updateStatus(id, "CONFIRMED");
    }

    public Contract findById(Long id) {
        return contractRepository.findById(id)
                .orElseThrow(() -> new IllegalArgumentException("계약을 찾을 수 없습니다."));
    }

    public long countAll() {
        return contractRepository.countAll();
    }

    private boolean isBlank(String value) {
        return value == null || value.trim().isEmpty();
    }
}

Writing /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/service/ContractService.java


In [28]:
%%writefile /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/service/InboundService.java

package com.example.wmspart4.service;

import com.example.wmspart4.domain.Contract;
import com.example.wmspart4.domain.Inbound;
import com.example.wmspart4.dto.InboundForm;
import com.example.wmspart4.repository.InboundRepository;
import com.example.wmspart4.repository.InventoryRepository;
import org.springframework.stereotype.Service;
import org.springframework.transaction.annotation.Transactional;

import java.util.List;

@Service
public class InboundService {

    private final InboundRepository inboundRepository;
    private final InventoryRepository inventoryRepository;
    private final ContractService contractService;

    public InboundService(InboundRepository inboundRepository,
                          InventoryRepository inventoryRepository,
                          ContractService contractService) {
        this.inboundRepository = inboundRepository;
        this.inventoryRepository = inventoryRepository;
        this.contractService = contractService;
    }

    public List<Inbound> findAll() {
        return inboundRepository.findAll();
    }

    public void register(InboundForm form) {
        if (form.getContractId() == null) {
            throw new IllegalArgumentException("계약을 선택해야 합니다.");
        }

        Contract contract = contractService.findById(form.getContractId());

        if (!contract.isConfirmed()) {
            throw new IllegalStateException("계약확정 상태의 계약만 입고 등록할 수 있습니다.");
        }

        if (form.getReceivedQuantity() == null || form.getReceivedQuantity() <= 0) {
            throw new IllegalArgumentException("입고 수량은 1 이상이어야 합니다.");
        }

        if (isBlank(form.getWarehouseName())) {
            throw new IllegalArgumentException("창고명을 입력해야 합니다.");
        }

        if (isBlank(form.getStorageZone())) {
            throw new IllegalArgumentException("보관구역을 입력해야 합니다.");
        }

        inboundRepository.save(form);
    }

    @Transactional
    public void complete(Long id) {
        Inbound inbound = inboundRepository.findById(id)
                .orElseThrow(() -> new IllegalArgumentException("입고 정보를 찾을 수 없습니다."));

        if (!inbound.isRegistered()) {
            throw new IllegalStateException("입고등록 상태에서만 완료 처리할 수 있습니다.");
        }

        Contract contract = contractService.findById(inbound.getContractId());

        inboundRepository.complete(id);
        inventoryRepository.createFromInbound(inbound, contract);
    }

    public long countAll() {
        return inboundRepository.countAll();
    }

    private boolean isBlank(String value) {
        return value == null || value.trim().isEmpty();
    }
}

Writing /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/service/InboundService.java


In [29]:
%%writefile /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/service/InventoryService.java

package com.example.wmspart4.service;

import com.example.wmspart4.domain.Inventory;
import com.example.wmspart4.repository.InventoryRepository;
import org.springframework.stereotype.Service;

import java.util.List;

@Service
public class InventoryService {

    private final InventoryRepository inventoryRepository;

    public InventoryService(InventoryRepository inventoryRepository) {
        this.inventoryRepository = inventoryRepository;
    }

    public List<Inventory> findAll() {
        return inventoryRepository.findAll();
    }

    public List<Inventory> findByCustomerId(Long customerId) {
        return inventoryRepository.findByCustomerId(customerId);
    }

    public Inventory findById(Long id) {
        return inventoryRepository.findById(id)
                .orElseThrow(() -> new IllegalArgumentException("재고를 찾을 수 없습니다."));
    }

    public long countAll() {
        return inventoryRepository.countAll();
    }
}


Writing /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/service/InventoryService.java


In [30]:
%%writefile /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/service/OutboundService.java

package com.example.wmspart4.service;

import com.example.wmspart4.domain.Inventory;
import com.example.wmspart4.domain.Outbound;
import com.example.wmspart4.dto.OutboundRequestForm;
import com.example.wmspart4.repository.InventoryRepository;
import com.example.wmspart4.repository.OutboundRepository;
import org.springframework.stereotype.Service;
import org.springframework.transaction.annotation.Transactional;

import java.util.List;

@Service
public class OutboundService {

    private final OutboundRepository outboundRepository;
    private final InventoryRepository inventoryRepository;

    public OutboundService(OutboundRepository outboundRepository,
                           InventoryRepository inventoryRepository) {
        this.outboundRepository = outboundRepository;
        this.inventoryRepository = inventoryRepository;
    }

    public void request(Long customerId, OutboundRequestForm form) {
        if (form.getInventoryId() == null) {
            throw new IllegalArgumentException("재고를 선택해야 합니다.");
        }

        Inventory inventory = inventoryRepository.findById(form.getInventoryId())
                .orElseThrow(() -> new IllegalArgumentException("재고를 찾을 수 없습니다."));

        if (!customerId.equals(inventory.getCustomerId())) {
            throw new IllegalStateException("본인 재고만 출고 요청할 수 있습니다.");
        }

        if (form.getRequestQuantity() == null || form.getRequestQuantity() <= 0) {
            throw new IllegalArgumentException("출고 수량은 1 이상이어야 합니다.");
        }

        if (form.getRequestQuantity() > inventory.getCurrentQuantity()) {
            throw new IllegalArgumentException("현재 재고 수량을 초과하여 출고 요청할 수 없습니다.");
        }

        outboundRepository.save(customerId, form);
    }

    public List<Outbound> findAll() {
        return outboundRepository.findAll();
    }

    public List<Outbound> findByCustomerId(Long customerId) {
        return outboundRepository.findByCustomerId(customerId);
    }

    @Transactional
    public void complete(Long id) {
        Outbound outbound = outboundRepository.findById(id)
                .orElseThrow(() -> new IllegalArgumentException("출고 요청을 찾을 수 없습니다."));

        if (!outbound.isRequested()) {
            throw new IllegalStateException("출고요청 상태에서만 완료 처리할 수 있습니다.");
        }

        Inventory inventory = inventoryRepository.findById(outbound.getInventoryId())
                .orElseThrow(() -> new IllegalArgumentException("재고를 찾을 수 없습니다."));

        if (outbound.getRequestQuantity() > inventory.getCurrentQuantity()) {
            throw new IllegalStateException("재고 수량이 부족합니다.");
        }

        outboundRepository.complete(id);
        inventoryRepository.decreaseQuantity(inventory.getId(), outbound.getRequestQuantity());
    }

    public long countAll() {
        return outboundRepository.countAll();
    }
}

Writing /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/service/OutboundService.java


In [31]:
%%writefile /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/service/NoticeService.java

package com.example.wmspart4.service;

import com.example.wmspart4.domain.Notice;
import com.example.wmspart4.dto.NoticeForm;
import com.example.wmspart4.repository.NoticeRepository;
import org.springframework.stereotype.Service;

import java.util.List;

@Service
public class NoticeService {

    private final NoticeRepository noticeRepository;

    public NoticeService(NoticeRepository noticeRepository) {
        this.noticeRepository = noticeRepository;
    }

    public void create(Long createdBy, NoticeForm form) {
        if (isBlank(form.getTitle()) || isBlank(form.getContent())) {
            throw new IllegalArgumentException("공지 제목과 내용을 입력해야 합니다.");
        }

        if (form.getVisible() == null) {
            form.setVisible(true);
        }

        noticeRepository.save(createdBy, form);
    }

    public List<Notice> findVisible() {
        return noticeRepository.findVisible();
    }

    public List<Notice> findAll() {
        return noticeRepository.findAll();
    }

    private boolean isBlank(String value) {
        return value == null || value.trim().isEmpty();
    }
}

Writing /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/service/NoticeService.java


In [32]:
%%writefile /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/service/InquiryService.java

package com.example.wmspart4.service;

import com.example.wmspart4.domain.Inquiry;
import com.example.wmspart4.dto.InquiryAnswerForm;
import com.example.wmspart4.dto.InquiryForm;
import com.example.wmspart4.repository.InquiryRepository;
import org.springframework.stereotype.Service;

import java.util.List;

@Service
public class InquiryService {

    private final InquiryRepository inquiryRepository;

    public InquiryService(InquiryRepository inquiryRepository) {
        this.inquiryRepository = inquiryRepository;
    }

    public void create(Long customerId, InquiryForm form) {
        if (isBlank(form.getTitle()) || isBlank(form.getContent())) {
            throw new IllegalArgumentException("문의 제목과 내용을 입력해야 합니다.");
        }

        inquiryRepository.save(customerId, form);
    }

    public void answer(Long id, InquiryAnswerForm form) {
        if (isBlank(form.getAnswerContent())) {
            throw new IllegalArgumentException("답변 내용을 입력해야 합니다.");
        }

        inquiryRepository.answer(id, form.getAnswerContent());
    }

    public List<Inquiry> findAll() {
        return inquiryRepository.findAll();
    }

    public List<Inquiry> findByCustomerId(Long customerId) {
        return inquiryRepository.findByCustomerId(customerId);
    }

    private boolean isBlank(String value) {
        return value == null || value.trim().isEmpty();
    }
}

Writing /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/service/InquiryService.java


In [33]:
%%writefile /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/config/SecurityConfig.java

package com.example.wmspart4.config;

import com.example.wmspart4.service.CustomUserDetailsService;
import org.springframework.context.annotation.Bean;
import org.springframework.context.annotation.Configuration;
import org.springframework.security.config.annotation.web.builders.HttpSecurity;
import org.springframework.security.core.Authentication;
import org.springframework.security.crypto.bcrypt.BCryptPasswordEncoder;
import org.springframework.security.crypto.password.PasswordEncoder;
import org.springframework.security.web.SecurityFilterChain;
import org.springframework.security.web.authentication.AuthenticationSuccessHandler;

@Configuration
public class SecurityConfig {

    private final CustomUserDetailsService customUserDetailsService;

    public SecurityConfig(CustomUserDetailsService customUserDetailsService) {
        this.customUserDetailsService = customUserDetailsService;
    }

    @Bean
    public SecurityFilterChain securityFilterChain(HttpSecurity http) throws Exception {
        http
                .userDetailsService(customUserDetailsService)
                .authorizeHttpRequests(auth -> auth
                        .requestMatchers("/style.css", "/login", "/signup").permitAll()
                        .requestMatchers("/admin/**").hasRole("ADMIN")
                        .requestMatchers("/customer/**").hasRole("CUSTOMER")
                        .anyRequest().authenticated()
                )
                .formLogin(form -> form
                        .loginPage("/login")
                        .loginProcessingUrl("/login")
                        .usernameParameter("email")
                        .passwordParameter("password")
                        .successHandler(roleBasedSuccessHandler())
                        .failureUrl("/login?error")
                        .permitAll()
                )
                .logout(logout -> logout
                        .logoutUrl("/logout")
                        .logoutSuccessUrl("/login?logout")
                        .invalidateHttpSession(true)
                        .deleteCookies("JSESSIONID")
                )
                .exceptionHandling(exception -> exception
                        .accessDeniedPage("/login?denied")
                );

        return http.build();
    }

    @Bean
    public PasswordEncoder passwordEncoder() {
        return new BCryptPasswordEncoder();
    }

    private AuthenticationSuccessHandler roleBasedSuccessHandler() {
        return (request, response, authentication) -> {
            if (hasRole(authentication, "ROLE_ADMIN")) {
                response.sendRedirect("/admin/dashboard");
                return;
            }

            if (hasRole(authentication, "ROLE_CUSTOMER")) {
                response.sendRedirect("/customer/dashboard");
                return;
            }

            response.sendRedirect("/login");
        };
    }

    private boolean hasRole(Authentication authentication, String role) {
        return authentication.getAuthorities().stream()
                .anyMatch(authority -> role.equals(authority.getAuthority()));
    }
}

Writing /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/config/SecurityConfig.java


In [34]:
%%writefile /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/controller/AuthController.java

package com.example.wmspart4.controller;

import com.example.wmspart4.dto.SignupForm;
import com.example.wmspart4.service.UserService;
import org.springframework.stereotype.Controller;
import org.springframework.ui.Model;
import org.springframework.web.bind.annotation.GetMapping;
import org.springframework.web.bind.annotation.PostMapping;

@Controller
public class AuthController {

    private final UserService userService;

    public AuthController(UserService userService) {
        this.userService = userService;
    }

    @GetMapping("/")
    public String home() {
        return "redirect:/login";
    }

    @GetMapping("/login")
    public String login() {
        return "login";
    }

    @GetMapping("/signup")
    public String signupForm(Model model) {
        model.addAttribute("signupForm", new SignupForm());
        return "signup";
    }

    @PostMapping("/signup")
    public String signup(SignupForm form, Model model) {
        try {
            userService.signup(form);
            return "redirect:/login?signup";
        } catch (IllegalArgumentException e) {
            model.addAttribute("signupForm", form);
            model.addAttribute("errorMessage", e.getMessage());
            return "signup";
        }
    }
}

Writing /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/controller/AuthController.java


In [35]:
%%writefile /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/controller/DashboardController.java

package com.example.wmspart4.controller;

import com.example.wmspart4.domain.AppUser;
import com.example.wmspart4.service.ContractService;
import com.example.wmspart4.service.InboundService;
import com.example.wmspart4.service.InventoryService;
import com.example.wmspart4.service.OutboundService;
import com.example.wmspart4.service.UserService;
import org.springframework.security.core.Authentication;
import org.springframework.stereotype.Controller;
import org.springframework.ui.Model;
import org.springframework.web.bind.annotation.GetMapping;

@Controller
public class DashboardController {

    private final UserService userService;
    private final ContractService contractService;
    private final InboundService inboundService;
    private final InventoryService inventoryService;
    private final OutboundService outboundService;

    public DashboardController(UserService userService,
                               ContractService contractService,
                               InboundService inboundService,
                               InventoryService inventoryService,
                               OutboundService outboundService) {
        this.userService = userService;
        this.contractService = contractService;
        this.inboundService = inboundService;
        this.inventoryService = inventoryService;
        this.outboundService = outboundService;
    }

    @GetMapping("/customer/dashboard")
    public String customerDashboard(Authentication authentication, Model model) {
        AppUser user = userService.findByEmail(authentication.getName());

        model.addAttribute("email", user.getEmail());
        model.addAttribute("name", user.getName());
        model.addAttribute("contracts", contractService.findByCustomerId(user.getId()).size());
        model.addAttribute("inventories", inventoryService.findByCustomerId(user.getId()).size());
        model.addAttribute("outbounds", outboundService.findByCustomerId(user.getId()).size());

        return "customer-dashboard";
    }

    @GetMapping("/admin/dashboard")
    public String adminDashboard(Authentication authentication, Model model) {
        model.addAttribute("email", authentication.getName());
        model.addAttribute("totalUsers", userService.countAllUsers());
        model.addAttribute("totalContracts", contractService.countAll());
        model.addAttribute("totalInbounds", inboundService.countAll());
        model.addAttribute("totalInventories", inventoryService.countAll());
        model.addAttribute("totalOutbounds", outboundService.countAll());

        return "admin-dashboard";
    }
}

Writing /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/controller/DashboardController.java


In [36]:
%%writefile /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/controller/ContractController.java

package com.example.wmspart4.controller;

import com.example.wmspart4.dto.ContractForm;
import com.example.wmspart4.service.ContractService;
import com.example.wmspart4.service.UserService;
import org.springframework.stereotype.Controller;
import org.springframework.ui.Model;
import org.springframework.web.bind.annotation.GetMapping;
import org.springframework.web.bind.annotation.PostMapping;
import org.springframework.web.bind.annotation.PathVariable;

@Controller
public class ContractController {

    private final ContractService contractService;
    private final UserService userService;

    public ContractController(ContractService contractService, UserService userService) {
        this.contractService = contractService;
        this.userService = userService;
    }

    @GetMapping("/admin/contracts")
    public String adminContracts(Model model) {
        model.addAttribute("contracts", contractService.findAll());
        model.addAttribute("customers", userService.findCustomers());
        model.addAttribute("contractForm", new ContractForm());
        return "admin-contracts";
    }

    @PostMapping("/admin/contracts")
    public String createContract(ContractForm form) {
        contractService.create(form);
        return "redirect:/admin/contracts";
    }

    @PostMapping("/admin/contracts/{id}/confirm")
    public String confirmContract(@PathVariable Long id) {
        contractService.confirm(id);
        return "redirect:/admin/contracts";
    }
}

Writing /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/controller/ContractController.java


In [37]:
%%writefile /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/controller/InboundController.java

package com.example.wmspart4.controller;

import com.example.wmspart4.dto.InboundForm;
import com.example.wmspart4.service.ContractService;
import com.example.wmspart4.service.InboundService;
import org.springframework.stereotype.Controller;
import org.springframework.ui.Model;
import org.springframework.web.bind.annotation.GetMapping;
import org.springframework.web.bind.annotation.PostMapping;
import org.springframework.web.bind.annotation.PathVariable;

@Controller
public class InboundController {

    private final InboundService inboundService;
    private final ContractService contractService;

    public InboundController(InboundService inboundService, ContractService contractService) {
        this.inboundService = inboundService;
        this.contractService = contractService;
    }

    @GetMapping("/admin/inbounds")
    public String adminInbounds(Model model) {
        model.addAttribute("inbounds", inboundService.findAll());
        model.addAttribute("contracts", contractService.findAll());
        model.addAttribute("inboundForm", new InboundForm());
        return "admin-inbounds";
    }

    @PostMapping("/admin/inbounds")
    public String registerInbound(InboundForm form) {
        inboundService.register(form);
        return "redirect:/admin/inbounds";
    }

    @PostMapping("/admin/inbounds/{id}/complete")
    public String completeInbound(@PathVariable Long id) {
        inboundService.complete(id);
        return "redirect:/admin/inbounds";
    }
}

Writing /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/controller/InboundController.java


In [38]:
%%writefile /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/controller/InventoryController.java

package com.example.wmspart4.controller;

import com.example.wmspart4.domain.AppUser;
import com.example.wmspart4.service.InventoryService;
import com.example.wmspart4.service.UserService;
import org.springframework.security.core.Authentication;
import org.springframework.stereotype.Controller;
import org.springframework.ui.Model;
import org.springframework.web.bind.annotation.GetMapping;

@Controller
public class InventoryController {

    private final InventoryService inventoryService;
    private final UserService userService;

    public InventoryController(InventoryService inventoryService, UserService userService) {
        this.inventoryService = inventoryService;
        this.userService = userService;
    }

    @GetMapping("/admin/inventories")
    public String adminInventories(Model model) {
        model.addAttribute("inventories", inventoryService.findAll());
        return "admin-inventories";
    }

    @GetMapping("/customer/inventories")
    public String customerInventories(Authentication authentication, Model model) {
        AppUser user = userService.findByEmail(authentication.getName());
        model.addAttribute("inventories", inventoryService.findByCustomerId(user.getId()));
        return "customer-inventories";
    }
}

Writing /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/controller/InventoryController.java


In [39]:
%%writefile /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/controller/OutboundController.java

package com.example.wmspart4.controller;

import com.example.wmspart4.domain.AppUser;
import com.example.wmspart4.dto.OutboundRequestForm;
import com.example.wmspart4.service.InventoryService;
import com.example.wmspart4.service.OutboundService;
import com.example.wmspart4.service.UserService;
import org.springframework.security.core.Authentication;
import org.springframework.stereotype.Controller;
import org.springframework.ui.Model;
import org.springframework.web.bind.annotation.GetMapping;
import org.springframework.web.bind.annotation.PostMapping;
import org.springframework.web.bind.annotation.PathVariable;

@Controller
public class OutboundController {

    private final OutboundService outboundService;
    private final InventoryService inventoryService;
    private final UserService userService;

    public OutboundController(OutboundService outboundService,
                              InventoryService inventoryService,
                              UserService userService) {
        this.outboundService = outboundService;
        this.inventoryService = inventoryService;
        this.userService = userService;
    }

    @GetMapping("/customer/outbounds")
    public String customerOutbounds(Authentication authentication, Model model) {
        AppUser user = userService.findByEmail(authentication.getName());

        model.addAttribute("outboundForm", new OutboundRequestForm());
        model.addAttribute("inventories", inventoryService.findByCustomerId(user.getId()));
        model.addAttribute("outbounds", outboundService.findByCustomerId(user.getId()));

        return "customer-outbounds";
    }

    @PostMapping("/customer/outbounds")
    public String requestOutbound(Authentication authentication, OutboundRequestForm form) {
        AppUser user = userService.findByEmail(authentication.getName());
        outboundService.request(user.getId(), form);
        return "redirect:/customer/outbounds";
    }

    @GetMapping("/admin/outbounds")
    public String adminOutbounds(Model model) {
        model.addAttribute("outbounds", outboundService.findAll());
        return "admin-outbounds";
    }

    @PostMapping("/admin/outbounds/{id}/complete")
    public String completeOutbound(@PathVariable Long id) {
        outboundService.complete(id);
        return "redirect:/admin/outbounds";
    }
}

Writing /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/controller/OutboundController.java


In [40]:
%%writefile /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/controller/NoticeController.java

package com.example.wmspart4.controller;

import com.example.wmspart4.domain.AppUser;
import com.example.wmspart4.dto.NoticeForm;
import com.example.wmspart4.service.NoticeService;
import com.example.wmspart4.service.UserService;
import org.springframework.security.core.Authentication;
import org.springframework.stereotype.Controller;
import org.springframework.ui.Model;
import org.springframework.web.bind.annotation.GetMapping;
import org.springframework.web.bind.annotation.PostMapping;

@Controller
public class NoticeController {

    private final NoticeService noticeService;
    private final UserService userService;

    public NoticeController(NoticeService noticeService, UserService userService) {
        this.noticeService = noticeService;
        this.userService = userService;
    }

    @GetMapping("/admin/notices")
    public String adminNotices(Model model) {
        model.addAttribute("notices", noticeService.findAll());
        model.addAttribute("noticeForm", new NoticeForm());
        model.addAttribute("isAdminPage", true);
        return "notices";
    }

    @PostMapping("/admin/notices")
    public String createNotice(Authentication authentication, NoticeForm form) {
        AppUser user = userService.findByEmail(authentication.getName());
        noticeService.create(user.getId(), form);
        return "redirect:/admin/notices";
    }

    @GetMapping("/customer/notices")
    public String customerNotices(Model model) {
        model.addAttribute("notices", noticeService.findVisible());
        model.addAttribute("isAdminPage", false);
        return "notices";
    }
}

Writing /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/controller/NoticeController.java


In [41]:
%%writefile /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/controller/InquiryController.java

package com.example.wmspart4.controller;

import com.example.wmspart4.domain.AppUser;
import com.example.wmspart4.dto.InquiryAnswerForm;
import com.example.wmspart4.dto.InquiryForm;
import com.example.wmspart4.service.InquiryService;
import com.example.wmspart4.service.UserService;
import org.springframework.security.core.Authentication;
import org.springframework.stereotype.Controller;
import org.springframework.ui.Model;
import org.springframework.web.bind.annotation.GetMapping;
import org.springframework.web.bind.annotation.PostMapping;
import org.springframework.web.bind.annotation.PathVariable;

@Controller
public class InquiryController {

    private final InquiryService inquiryService;
    private final UserService userService;

    public InquiryController(InquiryService inquiryService, UserService userService) {
        this.inquiryService = inquiryService;
        this.userService = userService;
    }

    @GetMapping("/customer/inquiries")
    public String customerInquiries(Authentication authentication, Model model) {
        AppUser user = userService.findByEmail(authentication.getName());

        model.addAttribute("inquiryForm", new InquiryForm());
        model.addAttribute("inquiries", inquiryService.findByCustomerId(user.getId()));

        return "customer-inquiries";
    }

    @PostMapping("/customer/inquiries")
    public String createInquiry(Authentication authentication, InquiryForm form) {
        AppUser user = userService.findByEmail(authentication.getName());
        inquiryService.create(user.getId(), form);
        return "redirect:/customer/inquiries";
    }

    @GetMapping("/admin/inquiries")
    public String adminInquiries(Model model) {
        model.addAttribute("answerForm", new InquiryAnswerForm());
        model.addAttribute("inquiries", inquiryService.findAll());

        return "admin-inquiries";
    }

    @PostMapping("/admin/inquiries/{id}/answer")
    public String answerInquiry(@PathVariable Long id, InquiryAnswerForm form) {
        inquiryService.answer(id, form);
        return "redirect:/admin/inquiries";
    }
}

Writing /content/spring-lab/wms-part4/src/main/java/com/example/wmspart4/controller/InquiryController.java


In [42]:
%%writefile /content/spring-lab/wms-part4/src/main/resources/templates/login.html

<!DOCTYPE html>
<html lang="ko" xmlns:th="http://www.thymeleaf.org">
<head>
    <meta charset="UTF-8">
    <title>WMS 로그인</title>
    <link rel="stylesheet" href="/style.css">
</head>
<body>
<div class="auth-page">
    <section class="auth-card">
        <p class="eyebrow">WMS PART IV</p>
        <h1>로그인</h1>
        <p class="description">WMS 통합 시스템에 접속한다.</p>

        <div class="message success" th:if="${param.signup}">
            회원가입이 완료되었습니다.
        </div>

        <div class="message error" th:if="${param.error}">
            이메일 또는 비밀번호가 올바르지 않습니다.
        </div>

        <div class="message error" th:if="${param.denied}">
            접근 권한이 없습니다.
        </div>

        <div class="message success" th:if="${param.logout}">
            로그아웃되었습니다.
        </div>

        <form method="post" action="/login">
            <input type="hidden" th:name="${_csrf.parameterName}" th:value="${_csrf.token}">

            <label>
                이메일
                <input type="email" name="email" placeholder="admin@test.com">
            </label>

            <label>
                비밀번호
                <input type="password" name="password" placeholder="1234">
            </label>

            <button type="submit">로그인</button>
        </form>

        <div class="auth-link">
            계정이 없으면 <a href="/signup">회원가입</a>
        </div>
    </section>
</div>
</body>
</html>


Writing /content/spring-lab/wms-part4/src/main/resources/templates/login.html


In [43]:
%%writefile /content/spring-lab/wms-part4/src/main/resources/templates/signup.html

<!DOCTYPE html>
<html lang="ko" xmlns:th="http://www.thymeleaf.org">
<head>
    <meta charset="UTF-8">
    <title>WMS 회원가입</title>
    <link rel="stylesheet" href="/style.css">
</head>
<body>
<div class="auth-page">
    <section class="auth-card">
        <p class="eyebrow">WMS PART IV</p>
        <h1>회원가입</h1>
        <p class="description">고객 또는 관리자 계정을 생성한다.</p>

        <div class="message error" th:if="${errorMessage != null}" th:text="${errorMessage}">
        </div>

        <form method="post" action="/signup" th:object="${signupForm}">
            <input type="hidden" th:name="${_csrf.parameterName}" th:value="${_csrf.token}">

            <label>
                이메일
                <input type="email" th:field="*{email}">
            </label>

            <label>
                비밀번호
                <input type="password" th:field="*{password}">
            </label>

            <label>
                이름
                <input type="text" th:field="*{name}">
            </label>

            <label>
                사용자 유형
                <select th:field="*{role}">
                    <option value="">선택</option>
                    <option value="ROLE_CUSTOMER">고객</option>
                    <option value="ROLE_ADMIN">관리자</option>
                </select>
            </label>

            <button type="submit">회원가입</button>
        </form>

        <div class="auth-link">
            이미 계정이 있으면 <a href="/login">로그인</a>
        </div>
    </section>
</div>
</body>
</html>

Writing /content/spring-lab/wms-part4/src/main/resources/templates/signup.html


In [44]:
%%writefile /content/spring-lab/wms-part4/src/main/resources/templates/admin-dashboard.html

<!DOCTYPE html>
<html lang="ko" xmlns:th="http://www.thymeleaf.org">
<head>
    <meta charset="UTF-8">
    <title>관리자 대시보드</title>
    <link rel="stylesheet" href="/style.css">
</head>
<body>
<div class="layout">
    <aside class="sidebar dark">
        <div class="brand">
            <div class="brand-mark">A</div>
            <div>
                <strong>WMS 관리자</strong>
                <span>Admin</span>
            </div>
        </div>

        <nav class="menu">
            <a class="active" href="/admin/dashboard">대시보드</a>
            <a href="/admin/contracts">계약 관리</a>
            <a href="/admin/inbounds">입고 관리</a>
            <a href="/admin/inventories">재고 관리</a>
            <a href="/admin/outbounds">출고 관리</a>
            <a href="/admin/notices">공지사항</a>
            <a href="/admin/inquiries">문의사항</a>
        </nav>

        <form method="post" action="/logout" class="logout-form">
            <input type="hidden" th:name="${_csrf.parameterName}" th:value="${_csrf.token}">
            <button type="submit">로그아웃</button>
        </form>
    </aside>

    <main class="content">
        <section class="hero">
            <p class="eyebrow">ADMIN DASHBOARD</p>
            <h1>관리자 대시보드</h1>
            <p th:text="${email} + ' 계정으로 로그인했습니다.'"></p>
        </section>

        <section class="summary-grid">
            <article>
                <span>사용자</span>
                <strong th:text="${totalUsers}">0</strong>
                <p>전체 계정</p>
            </article>

            <article>
                <span>계약</span>
                <strong th:text="${totalContracts}">0</strong>
                <p>전체 계약</p>
            </article>

            <article>
                <span>입고</span>
                <strong th:text="${totalInbounds}">0</strong>
                <p>입고 건수</p>
            </article>

            <article>
                <span>재고</span>
                <strong th:text="${totalInventories}">0</strong>
                <p>현재 재고</p>
            </article>

            <article>
                <span>출고</span>
                <strong th:text="${totalOutbounds}">0</strong>
                <p>출고 요청</p>
            </article>
        </section>
    </main>
</div>
</body>
</html>

Writing /content/spring-lab/wms-part4/src/main/resources/templates/admin-dashboard.html


In [45]:
%%writefile /content/spring-lab/wms-part4/src/main/resources/templates/customer-dashboard.html

<!DOCTYPE html>
<html lang="ko" xmlns:th="http://www.thymeleaf.org">
<head>
    <meta charset="UTF-8">
    <title>고객 대시보드</title>
    <link rel="stylesheet" href="/style.css">
</head>
<body>
<div class="layout">
    <aside class="sidebar">
        <div class="brand">
            <div class="brand-mark">C</div>
            <div>
                <strong>WMS 고객</strong>
                <span>Customer</span>
            </div>
        </div>

        <nav class="menu">
            <a class="active" href="/customer/dashboard">대시보드</a>
            <a href="/customer/inventories">재고 현황</a>
            <a href="/customer/outbounds">출고 요청</a>
            <a href="/customer/notices">공지사항</a>
            <a href="/customer/inquiries">문의사항</a>
        </nav>

        <form method="post" action="/logout" class="logout-form">
            <input type="hidden" th:name="${_csrf.parameterName}" th:value="${_csrf.token}">
            <button type="submit">로그아웃</button>
        </form>
    </aside>

    <main class="content">
        <section class="hero">
            <p class="eyebrow">CUSTOMER DASHBOARD</p>
            <h1>고객 대시보드</h1>
            <p th:text="${name} + '님, WMS에 접속했습니다.'"></p>
        </section>

        <section class="summary-grid customer-grid">
            <article>
                <span>내 계약</span>
                <strong th:text="${contracts}">0</strong>
                <p>계약 현황</p>
            </article>

            <article>
                <span>내 재고</span>
                <strong th:text="${inventories}">0</strong>
                <p>보관 재고</p>
            </article>

            <article>
                <span>내 출고</span>
                <strong th:text="${outbounds}">0</strong>
                <p>출고 요청</p>
            </article>
        </section>
    </main>
</div>
</body>
</html>


Writing /content/spring-lab/wms-part4/src/main/resources/templates/customer-dashboard.html


In [46]:
%%writefile /content/spring-lab/wms-part4/src/main/resources/templates/admin-contracts.html

<!DOCTYPE html>
<html lang="ko" xmlns:th="http://www.thymeleaf.org">
<head>
    <meta charset="UTF-8">
    <title>계약 관리</title>
    <link rel="stylesheet" href="/style.css">
</head>
<body>
<div class="layout">
    <aside class="sidebar dark">
        <div class="brand">
            <div class="brand-mark">A</div>
            <div>
                <strong>WMS 관리자</strong>
                <span>Admin</span>
            </div>
        </div>

        <nav class="menu">
            <a href="/admin/dashboard">대시보드</a>
            <a class="active" href="/admin/contracts">계약 관리</a>
            <a href="/admin/inbounds">입고 관리</a>
            <a href="/admin/inventories">재고 관리</a>
            <a href="/admin/outbounds">출고 관리</a>
            <a href="/admin/notices">공지사항</a>
            <a href="/admin/inquiries">문의사항</a>
        </nav>
    </aside>

    <main class="content">
        <section class="hero">
            <p class="eyebrow">CONTRACT</p>
            <h1>계약 관리</h1>
            <p>고객 계약을 등록하고 계약요청 상태의 계약을 확정한다.</p>
        </section>

        <section class="workspace">
            <div class="panel">
                <h2>계약 등록</h2>

                <form method="post" action="/admin/contracts" th:object="${contractForm}">
                    <input type="hidden" th:name="${_csrf.parameterName}" th:value="${_csrf.token}">

                    <label>
                        고객
                        <select th:field="*{customerId}">
                            <option value="">선택</option>
                            <option th:each="customer : ${customers}"
                                    th:value="${customer.id}"
                                    th:text="${customer.name}">
                            </option>
                        </select>
                    </label>

                    <label>
                        상품명
                        <input type="text" th:field="*{productName}">
                    </label>

                    <label>
                        수량
                        <input type="number" th:field="*{quantity}">
                    </label>

                    <label>
                        창고명
                        <input type="text" th:field="*{warehouseName}">
                    </label>

                    <label>
                        보관유형
                        <select th:field="*{storageType}">
                            <option value="">선택</option>
                            <option value="NORMAL">일반</option>
                            <option value="COLD">냉장</option>
                            <option value="FROZEN">냉동</option>
                        </select>
                    </label>

                    <label>
                        요청사항
                        <textarea th:field="*{requestMemo}"></textarea>
                    </label>

                    <button type="submit">계약 등록</button>
                </form>
            </div>

            <div class="panel wide">
                <h2>계약 목록</h2>

                <table>
                    <thead>
                    <tr>
                        <th>번호</th>
                        <th>고객</th>
                        <th>상품</th>
                        <th>수량</th>
                        <th>창고</th>
                        <th>보관유형</th>
                        <th>상태</th>
                        <th>처리</th>
                    </tr>
                    </thead>
                    <tbody>
                    <tr th:each="contract : ${contracts}">
                        <td th:text="${contract.id}"></td>
                        <td th:text="${contract.customerName}"></td>
                        <td th:text="${contract.productName}"></td>
                        <td th:text="${contract.quantity}"></td>
                        <td th:text="${contract.warehouseName}"></td>
                        <td th:text="${contract.storageTypeLabel}"></td>
                        <td>
                            <span class="status" th:text="${contract.statusLabel}"></span>
                        </td>
                        <td>
                            <form th:if="${contract.requested}"
                                  method="post"
                                  th:action="@{/admin/contracts/{id}/confirm(id=${contract.id})}">
                                <input type="hidden" th:name="${_csrf.parameterName}" th:value="${_csrf.token}">
                                <button type="submit">확정</button>
                            </form>

                            <span th:unless="${contract.requested}">처리완료</span>
                        </td>
                    </tr>
                    </tbody>
                </table>
            </div>
        </section>
    </main>
</div>
</body>
</html>

Writing /content/spring-lab/wms-part4/src/main/resources/templates/admin-contracts.html


In [47]:
%%writefile /content/spring-lab/wms-part4/src/main/resources/templates/admin-inbounds.html

<!DOCTYPE html>
<html lang="ko" xmlns:th="http://www.thymeleaf.org">
<head>
    <meta charset="UTF-8">
    <title>입고 관리</title>
    <link rel="stylesheet" href="/style.css">
</head>
<body>
<div class="layout">
    <aside class="sidebar dark">
        <div class="brand">
            <div class="brand-mark">A</div>
            <div>
                <strong>WMS 관리자</strong>
                <span>Admin</span>
            </div>
        </div>

        <nav class="menu">
            <a href="/admin/dashboard">대시보드</a>
            <a href="/admin/contracts">계약 관리</a>
            <a class="active" href="/admin/inbounds">입고 관리</a>
            <a href="/admin/inventories">재고 관리</a>
            <a href="/admin/outbounds">출고 관리</a>
            <a href="/admin/notices">공지사항</a>
            <a href="/admin/inquiries">문의사항</a>
        </nav>
    </aside>

    <main class="content">
        <section class="hero">
            <p class="eyebrow">INBOUND</p>
            <h1>입고 관리</h1>
            <p>확정된 계약을 기준으로 입고를 등록하고 입고완료를 처리한다.</p>
        </section>

        <section class="workspace">
            <div class="panel">
                <h2>입고 등록</h2>

                <form method="post" action="/admin/inbounds" th:object="${inboundForm}">
                    <input type="hidden" th:name="${_csrf.parameterName}" th:value="${_csrf.token}">

                    <label>
                        계약
                        <select th:field="*{contractId}">
                            <option value="">선택</option>
                            <option th:each="contract : ${contracts}"
                                    th:value="${contract.id}"
                                    th:text="${contract.id + ' - ' + contract.productName + ' / ' + contract.statusLabel}">
                            </option>
                        </select>
                    </label>

                    <label>
                        입고수량
                        <input type="number" th:field="*{receivedQuantity}">
                    </label>

                    <label>
                        창고명
                        <input type="text" th:field="*{warehouseName}">
                    </label>

                    <label>
                        보관구역
                        <input type="text" th:field="*{storageZone}">
                    </label>

                    <label>
                        파렛트번호
                        <input type="text" th:field="*{palletNo}">
                    </label>

                    <button type="submit">입고 등록</button>
                </form>
            </div>

            <div class="panel wide">
                <h2>입고 목록</h2>

                <table>
                    <thead>
                    <tr>
                        <th>번호</th>
                        <th>계약</th>
                        <th>고객</th>
                        <th>상품</th>
                        <th>수량</th>
                        <th>창고</th>
                        <th>구역</th>
                        <th>상태</th>
                        <th>처리</th>
                    </tr>
                    </thead>
                    <tbody>
                    <tr th:each="inbound : ${inbounds}">
                        <td th:text="${inbound.id}"></td>
                        <td th:text="${inbound.contractId}"></td>
                        <td th:text="${inbound.customerName}"></td>
                        <td th:text="${inbound.productName}"></td>
                        <td th:text="${inbound.receivedQuantity}"></td>
                        <td th:text="${inbound.warehouseName}"></td>
                        <td th:text="${inbound.storageZone}"></td>
                        <td>
                            <span class="status" th:text="${inbound.statusLabel}"></span>
                        </td>
                        <td>
                            <form th:if="${inbound.registered}"
                                  method="post"
                                  th:action="@{/admin/inbounds/{id}/complete(id=${inbound.id})}">
                                <input type="hidden" th:name="${_csrf.parameterName}" th:value="${_csrf.token}">
                                <button type="submit">입고완료</button>
                            </form>

                            <span th:unless="${inbound.registered}">완료</span>
                        </td>
                    </tr>
                    </tbody>
                </table>
            </div>
        </section>
    </main>
</div>
</body>
</html>

Writing /content/spring-lab/wms-part4/src/main/resources/templates/admin-inbounds.html


In [48]:
%%writefile /content/spring-lab/wms-part4/src/main/resources/templates/admin-inventories.html

<!DOCTYPE html>
<html lang="ko" xmlns:th="http://www.thymeleaf.org">
<head>
    <meta charset="UTF-8">
    <title>재고 관리</title>
    <link rel="stylesheet" href="/style.css">
</head>
<body>
<div class="layout">
    <aside class="sidebar dark">
        <div class="brand">
            <div class="brand-mark">A</div>
            <div>
                <strong>WMS 관리자</strong>
                <span>Admin</span>
            </div>
        </div>

        <nav class="menu">
            <a href="/admin/dashboard">대시보드</a>
            <a href="/admin/contracts">계약 관리</a>
            <a href="/admin/inbounds">입고 관리</a>
            <a class="active" href="/admin/inventories">재고 관리</a>
            <a href="/admin/outbounds">출고 관리</a>
            <a href="/admin/notices">공지사항</a>
            <a href="/admin/inquiries">문의사항</a>
        </nav>
    </aside>

    <main class="content">
        <section class="hero">
            <p class="eyebrow">INVENTORY</p>
            <h1>재고 관리</h1>
            <p>입고 완료로 생성된 전체 재고 수량과 보관 위치를 조회한다.</p>
        </section>

        <section class="panel wide">
            <table>
                <thead>
                <tr>
                    <th>번호</th>
                    <th>고객</th>
                    <th>상품</th>
                    <th>수량</th>
                    <th>창고</th>
                    <th>구역</th>
                    <th>파렛트</th>
                    <th>상태</th>
                </tr>
                </thead>
                <tbody>
                <tr th:each="inventory : ${inventories}">
                    <td th:text="${inventory.id}"></td>
                    <td th:text="${inventory.customerName}"></td>
                    <td th:text="${inventory.productName}"></td>
                    <td th:text="${inventory.currentQuantity}"></td>
                    <td th:text="${inventory.warehouseName}"></td>
                    <td th:text="${inventory.storageZone}"></td>
                    <td th:text="${inventory.palletNo}"></td>
                    <td>
                        <span class="status" th:text="${inventory.statusLabel}"></span>
                    </td>
                </tr>
                </tbody>
            </table>
        </section>
    </main>
</div>
</body>
</html>

Writing /content/spring-lab/wms-part4/src/main/resources/templates/admin-inventories.html


In [49]:
%%writefile /content/spring-lab/wms-part4/src/main/resources/templates/customer-inventories.html

<!DOCTYPE html>
<html lang="ko" xmlns:th="http://www.thymeleaf.org">
<head>
    <meta charset="UTF-8">
    <title>내 재고 현황</title>
    <link rel="stylesheet" href="/style.css">
</head>
<body>
<div class="layout">
    <aside class="sidebar">
        <div class="brand">
            <div class="brand-mark">C</div>
            <div>
                <strong>WMS 고객</strong>
                <span>Customer</span>
            </div>
        </div>

        <nav class="menu">
            <a href="/customer/dashboard">대시보드</a>
            <a class="active" href="/customer/inventories">재고 현황</a>
            <a href="/customer/outbounds">출고 요청</a>
            <a href="/customer/notices">공지사항</a>
            <a href="/customer/inquiries">문의사항</a>
        </nav>
    </aside>

    <main class="content">
        <section class="hero">
            <p class="eyebrow">MY INVENTORY</p>
            <h1>내 재고 현황</h1>
            <p>로그인한 고객 본인의 보관 재고만 조회한다.</p>
        </section>

        <section class="panel wide">
            <table>
                <thead>
                <tr>
                    <th>번호</th>
                    <th>상품</th>
                    <th>수량</th>
                    <th>창고</th>
                    <th>구역</th>
                    <th>파렛트</th>
                    <th>상태</th>
                </tr>
                </thead>
                <tbody>
                <tr th:each="inventory : ${inventories}">
                    <td th:text="${inventory.id}"></td>
                    <td th:text="${inventory.productName}"></td>
                    <td th:text="${inventory.currentQuantity}"></td>
                    <td th:text="${inventory.warehouseName}"></td>
                    <td th:text="${inventory.storageZone}"></td>
                    <td th:text="${inventory.palletNo}"></td>
                    <td>
                        <span class="status" th:text="${inventory.statusLabel}"></span>
                    </td>
                </tr>
                </tbody>
            </table>
        </section>
    </main>
</div>
</body>
</html>

Writing /content/spring-lab/wms-part4/src/main/resources/templates/customer-inventories.html


In [50]:
%%writefile /content/spring-lab/wms-part4/src/main/resources/templates/customer-outbounds.html

<!DOCTYPE html>
<html lang="ko" xmlns:th="http://www.thymeleaf.org">
<head>
    <meta charset="UTF-8">
    <title>출고 요청</title>
    <link rel="stylesheet" href="/style.css">
</head>
<body>
<div class="layout">
    <aside class="sidebar">
        <div class="brand">
            <div class="brand-mark">C</div>
            <div>
                <strong>WMS 고객</strong>
                <span>Customer</span>
            </div>
        </div>

        <nav class="menu">
            <a href="/customer/dashboard">대시보드</a>
            <a href="/customer/inventories">재고 현황</a>
            <a class="active" href="/customer/outbounds">출고 요청</a>
            <a href="/customer/notices">공지사항</a>
            <a href="/customer/inquiries">문의사항</a>
        </nav>
    </aside>

    <main class="content">
        <section class="hero">
            <p class="eyebrow">OUTBOUND</p>
            <h1>출고 요청</h1>
            <p>보관 중인 본인 재고를 선택하여 출고를 요청한다.</p>
        </section>

        <section class="workspace">
            <div class="panel">
                <h2>출고 요청 등록</h2>

                <form method="post" action="/customer/outbounds" th:object="${outboundForm}">
                    <input type="hidden" th:name="${_csrf.parameterName}" th:value="${_csrf.token}">

                    <label>
                        재고
                        <select th:field="*{inventoryId}">
                            <option value="">선택</option>
                            <option th:each="inventory : ${inventories}"
                                    th:value="${inventory.id}"
                                    th:text="${inventory.productName + ' / 현재수량 ' + inventory.currentQuantity}">
                            </option>
                        </select>
                    </label>

                    <label>
                        요청수량
                        <input type="number" th:field="*{requestQuantity}">
                    </label>

                    <label>
                        희망출고일
                        <input type="date" th:field="*{desiredDate}">
                    </label>

                    <label>
                        요청사항
                        <textarea th:field="*{requestMemo}"></textarea>
                    </label>

                    <button type="submit">출고 요청</button>
                </form>
            </div>

            <div class="panel wide">
                <h2>내 출고 현황</h2>

                <table>
                    <thead>
                    <tr>
                        <th>번호</th>
                        <th>상품</th>
                        <th>수량</th>
                        <th>상태</th>
                        <th>희망일</th>
                        <th>요청일시</th>
                    </tr>
                    </thead>
                    <tbody>
                    <tr th:each="outbound : ${outbounds}">
                        <td th:text="${outbound.id}"></td>
                        <td th:text="${outbound.productName}"></td>
                        <td th:text="${outbound.requestQuantity}"></td>
                        <td>
                            <span class="status" th:text="${outbound.statusLabel}"></span>
                        </td>
                        <td th:text="${outbound.desiredDate}"></td>
                        <td th:text="${outbound.requestedAt}"></td>
                    </tr>
                    </tbody>
                </table>
            </div>
        </section>
    </main>
</div>
</body>
</html>

Writing /content/spring-lab/wms-part4/src/main/resources/templates/customer-outbounds.html


In [51]:
%%writefile /content/spring-lab/wms-part4/src/main/resources/templates/admin-outbounds.html

<!DOCTYPE html>
<html lang="ko" xmlns:th="http://www.thymeleaf.org">
<head>
    <meta charset="UTF-8">
    <title>출고 관리</title>
    <link rel="stylesheet" href="/style.css">
</head>
<body>
<div class="layout">
    <aside class="sidebar dark">
        <div class="brand">
            <div class="brand-mark">A</div>
            <div>
                <strong>WMS 관리자</strong>
                <span>Admin</span>
            </div>
        </div>

        <nav class="menu">
            <a href="/admin/dashboard">대시보드</a>
            <a href="/admin/contracts">계약 관리</a>
            <a href="/admin/inbounds">입고 관리</a>
            <a href="/admin/inventories">재고 관리</a>
            <a class="active" href="/admin/outbounds">출고 관리</a>
            <a href="/admin/notices">공지사항</a>
            <a href="/admin/inquiries">문의사항</a>
        </nav>
    </aside>

    <main class="content">
        <section class="hero">
            <p class="eyebrow">ADMIN OUTBOUND</p>
            <h1>출고 관리</h1>
            <p>고객 출고 요청을 확인하고 출고완료를 처리한다.</p>
        </section>

        <section class="panel wide">
            <table>
                <thead>
                <tr>
                    <th>번호</th>
                    <th>고객</th>
                    <th>상품</th>
                    <th>수량</th>
                    <th>상태</th>
                    <th>희망일</th>
                    <th>처리</th>
                </tr>
                </thead>
                <tbody>
                <tr th:each="outbound : ${outbounds}">
                    <td th:text="${outbound.id}"></td>
                    <td th:text="${outbound.customerName}"></td>
                    <td th:text="${outbound.productName}"></td>
                    <td th:text="${outbound.requestQuantity}"></td>
                    <td>
                        <span class="status" th:text="${outbound.statusLabel}"></span>
                    </td>
                    <td th:text="${outbound.desiredDate}"></td>
                    <td>
                        <form th:if="${outbound.requested}"
                              method="post"
                              th:action="@{/admin/outbounds/{id}/complete(id=${outbound.id})}">
                            <input type="hidden" th:name="${_csrf.parameterName}" th:value="${_csrf.token}">
                            <button type="submit">출고완료</button>
                        </form>

                        <span th:unless="${outbound.requested}">완료</span>
                    </td>
                </tr>
                </tbody>
            </table>
        </section>
    </main>
</div>
</body>
</html>

Writing /content/spring-lab/wms-part4/src/main/resources/templates/admin-outbounds.html


In [52]:
%%writefile /content/spring-lab/wms-part4/src/main/resources/templates/notices.html

<!DOCTYPE html>
<html lang="ko" xmlns:th="http://www.thymeleaf.org">
<head>
    <meta charset="UTF-8">
    <title>공지사항</title>
    <link rel="stylesheet" href="/style.css">
</head>
<body>
<div class="layout">
    <aside class="sidebar" th:classappend="${isAdminPage} ? ' dark' : ''">
        <div class="brand">
            <div class="brand-mark" th:text="${isAdminPage} ? 'A' : 'C'">W</div>
            <div>
                <strong th:text="${isAdminPage} ? 'WMS 관리자' : 'WMS 고객'">WMS</strong>
                <span th:text="${isAdminPage} ? 'Admin' : 'Customer'">Role</span>
            </div>
        </div>

        <nav class="menu" th:if="${isAdminPage}">
            <a href="/admin/dashboard">대시보드</a>
            <a href="/admin/contracts">계약 관리</a>
            <a href="/admin/inbounds">입고 관리</a>
            <a href="/admin/inventories">재고 관리</a>
            <a href="/admin/outbounds">출고 관리</a>
            <a class="active" href="/admin/notices">공지사항</a>
            <a href="/admin/inquiries">문의사항</a>
        </nav>

        <nav class="menu" th:unless="${isAdminPage}">
            <a href="/customer/dashboard">대시보드</a>
            <a href="/customer/inventories">재고 현황</a>
            <a href="/customer/outbounds">출고 요청</a>
            <a class="active" href="/customer/notices">공지사항</a>
            <a href="/customer/inquiries">문의사항</a>
        </nav>
    </aside>

    <main class="content">
        <section class="hero">
            <p class="eyebrow">NOTICE</p>
            <h1>공지사항</h1>
            <p>WMS 운영 공지를 확인한다.</p>
        </section>

        <section class="panel" th:if="${isAdminPage}">
            <h2>공지 등록</h2>

            <form method="post" action="/admin/notices" th:object="${noticeForm}">
                <input type="hidden" th:name="${_csrf.parameterName}" th:value="${_csrf.token}">

                <label>
                    제목
                    <input type="text" th:field="*{title}">
                </label>

                <label>
                    내용
                    <textarea th:field="*{content}"></textarea>
                </label>

                <label>
                    표시여부
                    <select th:field="*{visible}">
                        <option value="true">표시</option>
                        <option value="false">숨김</option>
                    </select>
                </label>

                <button type="submit">공지 등록</button>
            </form>
        </section>

        <section class="panel wide">
            <h2>공지 목록</h2>

            <table>
                <thead>
                <tr>
                    <th>번호</th>
                    <th>제목</th>
                    <th>내용</th>
                    <th>표시</th>
                    <th>작성일</th>
                </tr>
                </thead>
                <tbody>
                <tr th:each="notice : ${notices}">
                    <td th:text="${notice.id}"></td>
                    <td th:text="${notice.title}"></td>
                    <td th:text="${notice.content}"></td>
                    <td th:text="${notice.visible} ? '표시' : '숨김'"></td>
                    <td th:text="${notice.createdAt}"></td>
                </tr>
                </tbody>
            </table>
        </section>
    </main>
</div>
</body>
</html>

Writing /content/spring-lab/wms-part4/src/main/resources/templates/notices.html


In [53]:
%%writefile /content/spring-lab/wms-part4/src/main/resources/templates/customer-inquiries.html

<!DOCTYPE html>
<html lang="ko" xmlns:th="http://www.thymeleaf.org">
<head>
    <meta charset="UTF-8">
    <title>문의사항</title>
    <link rel="stylesheet" href="/style.css">
</head>
<body>
<div class="layout">
    <aside class="sidebar">
        <div class="brand">
            <div class="brand-mark">C</div>
            <div>
                <strong>WMS 고객</strong>
                <span>Customer</span>
            </div>
        </div>

        <nav class="menu">
            <a href="/customer/dashboard">대시보드</a>
            <a href="/customer/inventories">재고 현황</a>
            <a href="/customer/outbounds">출고 요청</a>
            <a href="/customer/notices">공지사항</a>
            <a class="active" href="/customer/inquiries">문의사항</a>
        </nav>
    </aside>

    <main class="content">
        <section class="hero">
            <p class="eyebrow">INQUIRY</p>
            <h1>문의사항</h1>
            <p>문의 등록과 답변 상태를 확인한다.</p>
        </section>

        <section class="workspace">
            <div class="panel">
                <h2>문의 등록</h2>

                <form method="post" action="/customer/inquiries" th:object="${inquiryForm}">
                    <input type="hidden" th:name="${_csrf.parameterName}" th:value="${_csrf.token}">

                    <label>
                        제목
                        <input type="text" th:field="*{title}">
                    </label>

                    <label>
                        내용
                        <textarea th:field="*{content}"></textarea>
                    </label>

                    <button type="submit">문의 등록</button>
                </form>
            </div>

            <div class="panel wide">
                <h2>내 문의 목록</h2>

                <table>
                    <thead>
                    <tr>
                        <th>번호</th>
                        <th>제목</th>
                        <th>내용</th>
                        <th>상태</th>
                        <th>답변</th>
                    </tr>
                    </thead>
                    <tbody>
                    <tr th:each="inquiry : ${inquiries}">
                        <td th:text="${inquiry.id}"></td>
                        <td th:text="${inquiry.title}"></td>
                        <td th:text="${inquiry.content}"></td>
                        <td>
                            <span class="status" th:text="${inquiry.statusLabel}"></span>
                        </td>
                        <td th:text="${inquiry.answerContent}"></td>
                    </tr>
                    </tbody>
                </table>
            </div>
        </section>
    </main>
</div>
</body>
</html>

Writing /content/spring-lab/wms-part4/src/main/resources/templates/customer-inquiries.html


In [54]:
%%writefile /content/spring-lab/wms-part4/src/main/resources/templates/admin-inquiries.html

<!DOCTYPE html>
<html lang="ko" xmlns:th="http://www.thymeleaf.org">
<head>
    <meta charset="UTF-8">
    <title>문의 관리</title>
    <link rel="stylesheet" href="/style.css">
</head>
<body>
<div class="layout">
    <aside class="sidebar dark">
        <div class="brand">
            <div class="brand-mark">A</div>
            <div>
                <strong>WMS 관리자</strong>
                <span>Admin</span>
            </div>
        </div>

        <nav class="menu">
            <a href="/admin/dashboard">대시보드</a>
            <a href="/admin/contracts">계약 관리</a>
            <a href="/admin/inbounds">입고 관리</a>
            <a href="/admin/inventories">재고 관리</a>
            <a href="/admin/outbounds">출고 관리</a>
            <a href="/admin/notices">공지사항</a>
            <a class="active" href="/admin/inquiries">문의사항</a>
        </nav>
    </aside>

    <main class="content">
        <section class="hero">
            <p class="eyebrow">ADMIN INQUIRY</p>
            <h1>문의 관리</h1>
            <p>고객 문의를 확인하고 답변을 등록한다.</p>
        </section>

        <section class="panel wide">
            <table>
                <thead>
                <tr>
                    <th>번호</th>
                    <th>고객</th>
                    <th>제목</th>
                    <th>내용</th>
                    <th>상태</th>
                    <th>답변</th>
                </tr>
                </thead>
                <tbody>
                <tr th:each="inquiry : ${inquiries}">
                    <td th:text="${inquiry.id}"></td>
                    <td th:text="${inquiry.customerName}"></td>
                    <td th:text="${inquiry.title}"></td>
                    <td th:text="${inquiry.content}"></td>
                    <td>
                        <span class="status" th:text="${inquiry.statusLabel}"></span>
                    </td>
                    <td>
                        <form method="post"
                              th:action="@{/admin/inquiries/{id}/answer(id=${inquiry.id})}">
                            <input type="hidden" th:name="${_csrf.parameterName}" th:value="${_csrf.token}">
                            <input type="text" name="answerContent" placeholder="답변 입력">
                            <button type="submit">답변</button>
                        </form>
                    </td>
                </tr>
                </tbody>
            </table>
        </section>
    </main>
</div>
</body>
</html>

Writing /content/spring-lab/wms-part4/src/main/resources/templates/admin-inquiries.html


In [55]:
%%writefile /content/spring-lab/wms-part4/src/main/resources/static/style.css

* {
    box-sizing: border-box;
}

body {
    margin: 0;
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", "Noto Sans KR", sans-serif;
    background: #f5f7fb;
    color: #172033;
}

.auth-page {
    min-height: 100vh;
    display: flex;
    align-items: center;
    justify-content: center;
    padding: 40px;
}

.auth-card {
    width: 430px;
    background: #ffffff;
    border: 1px solid #e2e8f0;
    border-radius: 24px;
    padding: 34px;
    box-shadow: 0 16px 40px rgba(15, 23, 42, 0.08);
}

.layout {
    display: flex;
    min-height: 100vh;
}

.sidebar {
    width: 260px;
    background: #ffffff;
    border-right: 1px solid #e2e8f0;
    padding: 28px 24px;
}

.sidebar.dark {
    background: #0f172a;
    color: #ffffff;
}

.brand {
    display: flex;
    align-items: center;
    gap: 12px;
    margin-bottom: 42px;
}

.brand-mark {
    width: 44px;
    height: 44px;
    border-radius: 14px;
    background: #1d4ed8;
    color: #ffffff;
    display: flex;
    align-items: center;
    justify-content: center;
    font-weight: 800;
    font-size: 20px;
}

.sidebar.dark .brand-mark {
    background: #22c55e;
    color: #052e16;
}

.brand strong {
    display: block;
    font-size: 19px;
}

.brand span {
    display: block;
    color: #64748b;
    font-size: 13px;
    margin-top: 3px;
}

.sidebar.dark .brand span {
    color: #94a3b8;
}

.menu {
    display: flex;
    flex-direction: column;
    gap: 10px;
}

.menu a {
    padding: 14px 16px;
    border-radius: 12px;
    font-weight: 700;
    color: #475569;
    text-decoration: none;
}

.sidebar.dark .menu a {
    color: #cbd5e1;
}

.menu a.active {
    background: #1d4ed8;
    color: #ffffff;
}

.sidebar.dark .menu a.active {
    background: #22c55e;
    color: #052e16;
}

.logout-form {
    margin-top: 30px;
}

.logout-form button {
    width: 100%;
    background: #ef4444;
}

.content {
    flex: 1;
    padding: 36px;
}

.hero {
    background: #ffffff;
    border: 1px solid #e2e8f0;
    border-radius: 24px;
    padding: 34px;
    margin-bottom: 24px;
    box-shadow: 0 12px 30px rgba(15, 23, 42, 0.05);
}

.eyebrow {
    margin: 0 0 10px;
    color: #1d4ed8;
    font-weight: 800;
    letter-spacing: 0.08em;
}

.hero h1,
.auth-card h1 {
    margin: 0 0 12px;
    font-size: 32px;
    letter-spacing: -0.04em;
}

.hero p,
.description {
    color: #64748b;
    line-height: 1.7;
}

.summary-grid {
    display: grid;
    grid-template-columns: repeat(5, 1fr);
    gap: 18px;
    margin-bottom: 24px;
}

.summary-grid.customer-grid {
    grid-template-columns: repeat(3, 1fr);
}

.summary-grid article,
.panel {
    background: #ffffff;
    border: 1px solid #e2e8f0;
    border-radius: 20px;
    padding: 24px;
    box-shadow: 0 8px 24px rgba(15, 23, 42, 0.04);
}

.summary-grid span {
    color: #64748b;
    font-weight: 700;
}

.summary-grid strong {
    display: block;
    margin-top: 12px;
    font-size: 32px;
    color: #1d4ed8;
}

.summary-grid p {
    margin: 8px 0 0;
    color: #64748b;
    font-size: 14px;
}

.workspace {
    display: grid;
    grid-template-columns: 360px 1fr;
    gap: 24px;
}

.panel {
    margin-bottom: 24px;
}

.panel h2 {
    margin: 0 0 16px;
    font-size: 22px;
    letter-spacing: -0.03em;
}

.panel.wide {
    overflow-x: auto;
}

label {
    display: block;
    margin-bottom: 14px;
    font-weight: 800;
    color: #334155;
}

input,
select,
textarea {
    width: 100%;
    margin-top: 8px;
    border: 1px solid #cbd5e1;
    border-radius: 12px;
    padding: 12px 13px;
    font-size: 14px;
    font-family: inherit;
    color: #0f172a;
    background: #ffffff;
}

textarea {
    min-height: 90px;
    resize: vertical;
}

button {
    border: none;
    border-radius: 12px;
    padding: 11px 14px;
    font-weight: 800;
    cursor: pointer;
    font-family: inherit;
    background: #1d4ed8;
    color: #ffffff;
}

.auth-card button {
    width: 100%;
}

.auth-link {
    margin-top: 18px;
    text-align: center;
    color: #64748b;
}

.auth-link a {
    color: #1d4ed8;
    font-weight: 800;
    text-decoration: none;
}

.message {
    border-radius: 14px;
    padding: 12px 14px;
    margin-bottom: 16px;
    font-weight: 800;
}

.message.error {
    background: #fff1f2;
    color: #be123c;
    border: 1px solid #fecdd3;
}

.message.success {
    background: #ecfdf3;
    color: #047857;
    border: 1px solid #bbf7d0;
}

table {
    width: 100%;
    border-collapse: collapse;
    font-size: 14px;
}

th {
    text-align: left;
    background: #f1f5f9;
    color: #334155;
    padding: 12px;
    border-bottom: 1px solid #cbd5e1;
    white-space: nowrap;
}

td {
    padding: 12px;
    border-bottom: 1px solid #e2e8f0;
    color: #334155;
    vertical-align: middle;
    line-height: 1.55;
}

.status {
    display: inline-block;
    padding: 4px 10px;
    border-radius: 999px;
    font-weight: 800;
    background: #eff6ff;
    color: #1d4ed8;
    white-space: nowrap;
}

Writing /content/spring-lab/wms-part4/src/main/resources/static/style.css
